# 01 — ETL Pipeline

**Project:** Surgery duration prediction — Hillel Yaffe Medical Center  
**Input:** Raw data (כרגע קבצים ב-`data/raw/`, בעתיד SQL מהשרת)  
**Output:** Staged clean tables → merged central table for the ML model

## מבנה = פייפליין Python

המחברת בנויה כך שאפשר להעתיק ישירות לקבצים:

```
pipeline/
├── config.py       ← Block 1
├── extract.py      ← Block 2  (read_sql / read_excel)
├── transform.py    ← Block 3  (clean per table)
├── load.py         ← Block 4  (save staged output)
└── run.py          ← Block 5  (orchestrator — run one table at a time)
```

כל טבלה = 3 פונקציות: `extract_*` → `clean_*` → `save_*`  
בשרת: מחליפים את גוף `extract_*` ב-`pd.read_sql(query, conn)`.

## גישת העבודה

1. **Extract** — שליפה (SQL / קובץ)
2. **Profile** — בדיקה במחברת בלבד (לא עובר ל-production)
3. **Transform** — ניקוי
4. **Load** — שמירה ל-`data/clean/staged/`

> **כלל:** לא עוברים לטבלה הבאה בלי אישור מפורש.

## סדר הטבלאות

| # | טבלה | SQL table (עתידי) | מפתח | סטטוס |
|---|------|-------------------|------|--------|
| 1 | **ניתוחים** | `surgeries` | `מספר מקרה` + `Medical Record` | ✅ |
| 2 | **BMI** | `bmi` | `מספר מקרה` + `תאריך ניתוח` | ✅ |
| 3 | **עישון** | `smoking` | `מספר מקרה` + `מועד ניתוח` | ✅ |
| 4 | **לחץ דם** | `blood_pressure` | `מספר מקרה` + `תאריך ניתוח` | 🔄 |
| 5 | **סטורציה** | `saturation` | `מספר מקרה` + `תאריך ניתוח` | 🔄 |
| 6 | **הרדמה** | `anesthesia` | `מספר מקרה` | ⏸ |
| 7 | **תרופות בניתוח** | `surgery_meds` | `מספר מקרה` | 🔄 |
| 8 | **מחלות רקע** | `background_diseases` | `patient` | 🔄 |
| 9 | רגישות אחרת | `other_allergy` | `patient` | ❌ הוסר |
| 10 | רגישות לתרופות | `drug_allergy` | `patient` | ✅ |
| 11 | תרופות קבועות | | `patient` | ❌ הוסר |
| 12 | **בדיקות מעבדה** | `labs` | `מספר מקרה` | 🔄 |

---
## Block 1 — config.py

נתיבים, שמות טבלאות, ושאילתות SQL (placeholder).

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

# --- paths ---
RAW_DIR = Path("data/raw")
CLEAN_DIR = Path("data/clean")
STAGED_DIR = CLEAN_DIR / "staged"
STAGED_DIR.mkdir(parents=True, exist_ok=True)

# --- local files (dev only — replace with SQL on server) ---
FILES = {
    "surgeries": "ניתוחים_260415.xlsx",
    "bmi": "סוג דם גובה משקל BMI.xlsx",
    "smoking": "עישון.xlsx",
    "bp": "לחץ דם.xlsx",
    "saturation": "סטורציה.xlsx",
    "surgery_meds": "תרופות בניתוח.xlsx",
    "labs": "בדיקות מעבדה.csv",
    "background_diseases": "מחלות רקע.xlsx",
    "other_allergy": "רגישות אחרת.xlsx",
    "drug_allergy": "רגישות לתרופות.xlsx",
    "chronic_meds": "תרופות קבועות.xlsx",
    "anesthesia": "הרדמה.xlsx",
}

LAB_COLUMNS = [
    "Patient", "מספר מקרה", "קטגוריית בדיקה",
    "שם בדיקה", "ערך", "תאריך בדיקה",
]

# --- SQL queries (production) ---
SQL = {
    "surgeries": "SELECT * FROM surgeries",
    "bmi": "SELECT * FROM bmi",
    "smoking": "SELECT * FROM smoking",
    "bp": "SELECT * FROM blood_pressure",
    "saturation": "SELECT * FROM saturation",
    "background_diseases": "SELECT * FROM background_diseases",
    "drug_allergy": "SELECT * FROM drug_allergy",
    "other_allergy": "SELECT * FROM other_allergy",
    "chronic_meds": "SELECT * FROM chronic_meds",
    "surgery_meds": "SELECT * FROM surgery_meds",
    "labs": "SELECT * FROM labs",
    # add rest when moving to server
}

# --- surgeries cleaning rules ---
SURGERIES_GROUP_KEYS = ["מספר מקרה", "Medical Record"]

SURGERIES_COLS_TO_DROP = [
    "פעולה ניתוחית מיילדותית", "F45", "F46", "הרדמה פירוט",
    "מחלקה מנתחת נוספת", "הערה לפרוצדורה", "הערה לאבחנה",
    "עוזר מנתח", "מנתח ראשי", "מרדים", "אחות רחוצה", "מנתח אחראי","אחות מסתובבת"
]

# --- total_surgery_time (OR occupancy, minutes) ---
ENTRY_DATE_COL = "תאריך כניסה לחדר ניתוח"
ENTRY_TIME_COL = "שעת כניסה לחדר ניתוח"
ANES_START_DATE = "תאריך תחילת הרדמה"
ANES_START_TIME = "שעת תחילת הרדמה"
SURG_START_DATE = "תאריך תחילת ניתוח"
SURG_START_TIME = "שעת תחילת ניתוח"
EXIT_DATE_COL = "תאריך יציאה מחדר ניתוח"
EXIT_TIME_COL = "שעת יציאה מחדר ניתוח"
ANES_END_DATE = "תאריך סיום הרדמה"
ANES_END_TIME = "שעת סיום הרדמה"
SURG_END_DATE = "תאריך סיום ניתוח"
SURG_END_TIME = "שעת סיום ניתוח"
DISC_DATE_COL = "תאריך שחרור מח ניתוח"
DISC_TIME_COL = "שעת שחרור מח ניתוח"
TARGET_COL = "total_surgery_time"
MAX_START_GAP_HOURS = 2
MAX_END_GAP_HOURS = 2
DISCHARGE_CLOSE_MINUTES = 30
SURGERY_DATE_COL = "תאריך ניתוח"

SURGERIES_DROP_IF_MISSING = [TARGET_COL, "גיל"]
SURGERIES_FILL_UNKNOWN = [
    "סוג ניתוח", "סוג הרדמה",
    "קוד אבחנה", "שם אבחנה", "קוד פרוצדורה", "שם פרוצדורה",
]
SURGERIES_COLS_TO_DROP_EXTRA = ["חדר"]
SURGERIES_FILL_NON = ["צד פרוצדורה"]
PROCEDURE_SIDE_COL = "צד פרוצדורה"
SURGERIES_INT_COLS = ["מספר מקרה", "גיל", TARGET_COL]

SURGERIES_RAW_TIMESTAMP_COLS = [
    ENTRY_DATE_COL, ENTRY_TIME_COL,
    ANES_START_DATE, ANES_START_TIME,
    SURG_START_DATE, SURG_START_TIME,
    EXIT_DATE_COL, EXIT_TIME_COL,
    ANES_END_DATE, ANES_END_TIME,
    SURG_END_DATE, SURG_END_TIME,
    DISC_DATE_COL, DISC_TIME_COL,
]

SURGERIES_COMPUTED_DATETIME_COLS = [
    "entry_to_or_datetime", "anesthesia_start_datetime", "surgery_start_datetime",
    "exit_from_or_datetime", "anesthesia_end_datetime", "surgery_end_datetime",
    "discharge_datetime", "selected_start_datetime", "selected_end_datetime",
]

# --- BMI cleaning rules ---
BMI_GROUP_KEYS = ["מספר מקרה", "תאריך ניתוח"]
BMI_COLS_TO_DROP = ["דם"]
MERGED_SURGERIES_NAME = "surgeries_with_bmi"
BMI_IMPUTE_MAX_AGE_K = 10  # widen age window up to ±10 years

# --- smoking cleaning rules ---
SMOKING_GROUP_KEYS = ["מספר מקרה", "מועד ניתוח"]
SMOKING_COL = "Smoking"
SMOKING_FILL_UNKNOWN = "UNKNOWN"
MERGED_AFTER_SMOKING_NAME = "surgeries_with_bmi_smoking"

# --- one-hot encoding ---
OHE_CATEGORICAL_COLS = ["מין", "סוג מקרה", "יום ניתוח", "סוג ניתוח", "סוג הרדמה"]
ANESTHESIA_TYPE_COL = "סוג הרדמה"
SURGERY_TYPE_COL = "סוג ניתוח"
SMOKING_OHE_PREFIX = "עישון"
SMOKER_LABEL = "מעשן"
NON_SMOKER_LABEL = "לא מעשן"

# --- blood pressure ---
BP_CASE_COL_RAW = "מקרה"
BP_GROUP_KEYS = ["מספר מקרה", "תאריך ניתוח"]
BP_SBP_COL = "היסטולי לפני ניתוח"
BP_DBP_COL = "דיאסטולי לפני ניתוח"

# --- saturation ---
SAT_CASE_COL_RAW = "מקרה"
SAT_GROUP_KEYS = ["מספר מקרה", "תאריך ניתוח"]
SAT_COL = "סטורציה לפני ניתוח"

# --- patient-level tables ---
PATIENT_KEY = "patient"
BG_OUTPUT_COL = "מחלות_רקע"
DRUG_ALLERGY_OUTPUT_COL = "רגישות_לתרופות"
OTHER_ALLERGY_OUTPUT_COL = "רגישות_אחרת"
CHRONIC_MEDS_OUTPUT_COL = "תרופות_קבועות"

# --- surgery meds ---
SURGERY_MEDS_KEY = "מספר מקרה"
SURGERY_MEDS_VALUE_COL = "תרופה"
SURGERY_MEDS_OUTPUT_COL = "תרופות_בניתוח"
SURGERY_MEDS_DROP_COLS = ["אופן מתן"]

# --- staged merge outputs ---
MERGED_AFTER_BP_NAME = "surgeries_with_bmi_smoking_bp"
MERGED_AFTER_SAT_NAME = "surgeries_with_bmi_smoking_bp_sat"
MERGED_AFTER_PATIENT_NAME = "surgeries_with_bmi_smoking_bp_sat_patient"
MERGED_AFTER_SURGERY_MEDS_NAME = "surgeries_with_bmi_smoking_bp_sat_patient_meds"

# --- feature engineering ---
COLS_TO_DROP = ["רגישות_אחרת", "תרופות_קבועות", "חדר"]
AGE_YOUNG_MAX = 18
AGE_ELDERLY_MIN = 65
BMI_UNDERWEIGHT_MAX = 18.5
BMI_NORMAL_MAX = 25.0
BMI_OVERWEIGHT_MAX = 30.0

# --- drugs index ---
DRUGS_DIR = Path("data/Drugs")
DRUGS_OUTPUT_DIR = DRUGS_DIR / "Output"
DRUGS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRUG_INDEX_PATH = DRUGS_DIR / "unique_drug_names.csv"
CLEANED_DRUGS_COL = "CLEANED_DRUGS"
CLEANED_DRUG_ALLERGIES_COL = "CLEANED_DRUG_ALLERGIES"
CLEANED_DRUGS_EXPORT_PATH = DRUGS_OUTPUT_DIR / "cleaned_drugs_comparison.csv"
BG_COUNT_COL = "מחלות_רקע_count"
DRUG_ALLERGY_COUNT_COL = "רגישות_לתרופות_count"
PROCEDURE_CODE_COUNT_COL = "קוד_פרוצדורה_count"

# --- labs ---
LAB_DATE_COL = "תאריך בדיקה"
LAB_TEST_COL = "שם בדיקה"
LAB_VALUE_COL = "ערך"
LABS_FILL_VALUE = 0
LAB_SELECTED_TESTS = [
    "ALT(GPT)-B", "AST(GOT)-B", "Albumin-B", "Albumin/Globulin-B",
    "Alkaline Phosphatase-B", "Amylase-B", "BASO abs", "BASO%", "BUN-B",
    "Bilirubin, total-B", "CK-B", "CRP", "Calcium-B", "Creatinine-B",
    "EOS abs", "EOS%", "GGT- Gamma Glutam.Trans.-B", "Globulin - blood",
    "Glucose-B", "HCT", "HGB", "Hemolysis Index -B", "Icteric Index-B",
    "LD Lactate dehydrogenase-B", "LYMPHO abs", "LYMPHO%", "Lipemic Index-B",
    "MCH", "MCHC", "MCV", "MONO abs", "MONO%", "MPV", "Magnesium-B",
    "NEUTRO abs", "NEUTRO%", "NRBC  abs",
    "NRBC (Nucleated red blood cells) / leukocytes % - blood",
    "OSM-cal-B", "PCT", "PDW", "PT,SEC", "PT-INR-B", "PTT-blood",
    "Phosphor -B", "Platelet, automated count - blood", "Potassium-B",
    "Protein, total-B", "RBC", "RDW", "Sodium-B", "Uric acid-B", "WBC",
]
MERGED_AFTER_LABS_NAME = "surgeries_merged_final"

# --- final dataset normalization ---
ANESTHESIA_OHE_PREFIX = "סוג הרדמה"
SURGERY_TYPE_OHE_PREFIX = "סוג ניתוח"
OHE_DROP_UNKNOWN_COLS = [
    f"{SURGERY_TYPE_OHE_PREFIX}_UNKNOWN",
    f"{ANESTHESIA_OHE_PREFIX}_UNKNOWN",
]
SAT_NORM_COL = "oxygen_saturation_norm"
SBP_Z_COL = "systolic_bp_Z"
DBP_Z_COL = "diastolic_bp_Z"
BMI_Z_COL = "BMI_Z"
SMOKER_COL = "smoker"

# --- English ML dataset export ---
BG_TOP_K_ICD9 = 65
BG_ICD9_OHE_PREFIX = "ICD9"
ESTIMATED_SURGERY_TIME_COL = "Estimated_Surgery_Time"
SURGERY_TIME_ANOMALY_COL = "Surgery_Time_Anomaly"
NUM_MEDICATIONS_COL = "Num_Medications"
NUM_ALLERGY_MEDICATIONS_COL = "Num_Allergy_Medications"
ML_DATASET_NAME = "surgeries_ml_ready"
SURGICAL_DEPARTMENT_COL = "Surgical_Department"
ML_EXPORT_DIR = CLEAN_DIR / "export"
ML_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

DEPARTMENT_EXPORT_FILES = {
    "אורטופדיה": "surgeries_ml_ready_orthopedics.csv",
    "כירורגיה": "surgeries_ml_ready_general_surgery.csv",
    "אף אוזן גרון": "surgeries_ml_ready_ent.csv",
}

ML_COLUMNS_DROP = [
    "שם פרוצדורה",
    "שם אבחנה",
    "start_time_source",
    "end_time_source",
    "duration_quality",
    BP_SBP_COL,
    BP_DBP_COL,
    SAT_COL,
    "קוד אבחנה",
    "גובה",
    "משקל",
    BG_COUNT_COL,
    PROCEDURE_CODE_COUNT_COL,
]

# notebook display only
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 200)

---
## Block 2 — extract.py

פונקציית extract לכל טבלה.  
כרגע: `read_excel` / `read_csv`.  
בשרת: `pd.read_sql(SQL["table_name"], conn)`.

In [2]:
def extract_surgeries(conn=None):
    """Pull surgeries table from DB or local file."""
    if conn is not None:
        return pd.read_sql(SQL["surgeries"], conn)
    return pd.read_excel(RAW_DIR / FILES["surgeries"])


def extract_bmi(conn=None):
    """Pull BMI table from DB or local file."""
    if conn is not None:
        return pd.read_sql(SQL["bmi"], conn)
    return pd.read_excel(RAW_DIR / FILES["bmi"])


def extract_smoking(conn=None):
    """Pull smoking table from DB or local file."""
    if conn is not None:
        return pd.read_sql(SQL["smoking"], conn)
    return pd.read_excel(RAW_DIR / FILES["smoking"])


def extract_bp(conn=None):
    if conn is not None:
        return pd.read_sql(SQL["bp"], conn)
    return pd.read_excel(RAW_DIR / FILES["bp"])


def extract_saturation(conn=None):
    if conn is not None:
        return pd.read_sql(SQL["saturation"], conn)
    return pd.read_excel(RAW_DIR / FILES["saturation"])


def extract_background_diseases(conn=None):
    if conn is not None:
        return pd.read_sql(SQL["background_diseases"], conn)
    return pd.read_excel(RAW_DIR / FILES["background_diseases"])


def extract_drug_allergy(conn=None):
    if conn is not None:
        return pd.read_sql(SQL["drug_allergy"], conn)
    return pd.read_excel(RAW_DIR / FILES["drug_allergy"])


def extract_other_allergy(conn=None):
    if conn is not None:
        return pd.read_sql(SQL["other_allergy"], conn)
    return pd.read_excel(RAW_DIR / FILES["other_allergy"])


def extract_chronic_meds(conn=None):
    if conn is not None:
        return pd.read_sql(SQL["chronic_meds"], conn)
    return pd.read_excel(RAW_DIR / FILES["chronic_meds"])


def extract_surgery_meds(conn=None):
    if conn is not None:
        return pd.read_sql(SQL["surgery_meds"], conn)
    return pd.read_excel(RAW_DIR / FILES["surgery_meds"])


def extract_labs(conn=None):
    """Pull labs table from DB or local CSV (no header)."""
    if conn is not None:
        return pd.read_sql(SQL["labs"], conn)
    return pd.read_csv(RAW_DIR / FILES["labs"], header=None, names=LAB_COLUMNS)


---
## Block 3 — transform.py

פונקציית `clean_*` לכל טבלה.  
מקבלת DataFrame גולמי, מחזירה DataFrame נקי.

In [3]:
def map_parent_department(dept_series):
    """אורטופדיה א/ב -> אורטופדיה, כירורגיה א/ב -> כירורגיה, אף אוזן גרון נשאר."""
    dept = dept_series.str.strip()
    return dept.str.replace(r" [אב]$", "", regex=True)


def join_if_many(series):
    """ערך יחיד -> נשמר כמו שהוא. כמה ערכים -> מחרוזת מופרדת בפסיק."""
    values = series.dropna()
    if len(values) == 0:
        return pd.NA
    unique = values.unique()
    if len(unique) == 1:
        return unique[0]
    return ",".join(str(v) for v in unique)


def group_to_one_row_per_surgery(df, group_keys):
    """שורה אחת לכל (מספר מקרה, Medical Record)."""
    other_cols = [c for c in df.columns if c not in group_keys]
    agg = {col: join_if_many for col in other_cols}
    return df.groupby(group_keys, as_index=False).agg(agg)


def build_datetime(date_series, time_series):
    """Combine date + time into one pandas datetime."""
    date_part = pd.to_datetime(date_series, errors="coerce")
    time_part = pd.to_timedelta(time_series.astype(str), errors="coerce")
    return date_part + time_part


def _is_consistent_with_others(ts, others, max_gap):
    """True if ts is not null and within max_gap of every other non-null timestamp."""
    if pd.isna(ts):
        return False
    for other in others:
        if pd.isna(other):
            continue
        if abs(ts - other) > max_gap:
            return False
    return True


def choose_consistent_start(entry, anes_start, surg_start, max_gap_hours=2):
    """Pick earliest valid OR start timestamp with cross-check against other starts."""
    max_gap = pd.Timedelta(hours=max_gap_hours)
    selected = pd.Series(pd.NaT, index=entry.index, dtype="datetime64[ns]")
    source = pd.Series("missing_start", index=entry.index, dtype=object)

    candidates = [
        ("entry_to_or", entry),
        ("anesthesia_start", anes_start),
        ("surgery_start", surg_start),
    ]

    for i in entry.index:
        row_vals = [(name, candidates[j][1].loc[i]) for j, (name, _) in enumerate(candidates)]
        if not any(not pd.isna(ts) for _, ts in row_vals):
            continue

        picked = False
        for name, ts in row_vals:
            others = [t for n, t in row_vals if n != name]
            if _is_consistent_with_others(ts, others, max_gap):
                selected.loc[i] = ts
                source.loc[i] = name
                picked = True
                break

        if not picked:
            source.loc[i] = "invalid_start"

    return selected, source


def choose_consistent_end(exit_or, anes_end, surg_end, discharge, max_gap_hours=2, discharge_close_minutes=30):
    """Pick latest valid OR end timestamp; discharge only if close to selected end."""
    max_gap = pd.Timedelta(hours=max_gap_hours)
    close_window = pd.Timedelta(minutes=discharge_close_minutes)
    selected = pd.Series(pd.NaT, index=exit_or.index, dtype="datetime64[ns]")
    source = pd.Series("missing_end", index=exit_or.index, dtype=object)

    candidates = [
        ("exit_from_or", exit_or),
        ("anesthesia_end", anes_end),
        ("surgery_end", surg_end),
    ]

    for i in exit_or.index:
        row_vals = [(name, candidates[j][1].loc[i]) for j, (name, _) in enumerate(candidates)]
        disc = discharge.loc[i]

        if not any(not pd.isna(ts) for _, ts in row_vals) and pd.isna(disc):
            continue

        base_ts = pd.NaT
        base_src = "missing_end"
        for name, ts in row_vals:
            others = [t for n, t in row_vals if n != name]
            if _is_consistent_with_others(ts, others, max_gap):
                base_ts = ts
                base_src = name
                break

        if pd.isna(base_ts):
            if any(not pd.isna(ts) for _, ts in row_vals):
                source.loc[i] = "invalid_end"
            continue

        if not pd.isna(disc) and abs(disc - base_ts) <= close_window:
            selected.loc[i] = disc
            source.loc[i] = "discharge_close_to_end"
        else:
            selected.loc[i] = base_ts
            source.loc[i] = base_src

    return selected, source


def _classify_duration(start_dt, end_dt, duration_min):
    if pd.isna(start_dt):
        return "missing_start"
    if pd.isna(end_dt):
        return "missing_end"
    if pd.isna(duration_min) or duration_min <= 0:
        return "negative_or_zero_duration"
    if duration_min > 24 * 60:
        return "over_24h"
    if duration_min > 12 * 60:
        return "suspicious_over_12h"
    return "valid"


def calculate_total_surgery_time(df):
    """OR occupancy time in minutes + quality flags. Does not drop rows."""
    df = df.copy()

    # 1. build all datetime columns
    df["entry_to_or_datetime"] = build_datetime(df[ENTRY_DATE_COL], df[ENTRY_TIME_COL])
    df["anesthesia_start_datetime"] = build_datetime(df[ANES_START_DATE], df[ANES_START_TIME])
    df["surgery_start_datetime"] = build_datetime(df[SURG_START_DATE], df[SURG_START_TIME])
    df["exit_from_or_datetime"] = build_datetime(df[EXIT_DATE_COL], df[EXIT_TIME_COL])
    df["anesthesia_end_datetime"] = build_datetime(df[ANES_END_DATE], df[ANES_END_TIME])
    df["surgery_end_datetime"] = build_datetime(df[SURG_END_DATE], df[SURG_END_TIME])
    df["discharge_datetime"] = build_datetime(df[DISC_DATE_COL], df[DISC_TIME_COL])

    # 2. pick consistent start / end
    start_dt, start_src = choose_consistent_start(
        df["entry_to_or_datetime"],
        df["anesthesia_start_datetime"],
        df["surgery_start_datetime"],
        max_gap_hours=MAX_START_GAP_HOURS,
    )
    end_dt, end_src = choose_consistent_end(
        df["exit_from_or_datetime"],
        df["anesthesia_end_datetime"],
        df["surgery_end_datetime"],
        df["discharge_datetime"],
        max_gap_hours=MAX_END_GAP_HOURS,
        discharge_close_minutes=DISCHARGE_CLOSE_MINUTES,
    )

    df["selected_start_datetime"] = start_dt
    df["selected_end_datetime"] = end_dt
    df["start_time_source"] = start_src
    df["end_time_source"] = end_src

    # 3. duration in minutes (midnight crossing handled by full datetimes)
    duration_min = (end_dt - start_dt).dt.total_seconds() / 60

    quality = []
    total = []
    for s, e, d in zip(start_dt, end_dt, duration_min):
        q = _classify_duration(s, e, d)
        quality.append(q)
        if q in ("missing_start", "missing_end", "negative_or_zero_duration", "over_24h"):
            total.append(pd.NA)
        else:
            total.append(d)

    df["duration_quality"] = quality
    df[TARGET_COL] = pd.to_numeric(total, errors="coerce")

    return df


def handle_surgeries_missing_values(df):
    """Fill categoricals; drop rows missing target or age."""
    df = df.copy()

    for col in SURGERIES_FILL_UNKNOWN:
        if col in df.columns:
            df[col] = df[col].fillna("UNKNOWN")

    for col in SURGERIES_FILL_NON:
        if col in df.columns:
            df[col] = df[col].fillna("NON")

    for col in SURGERIES_DROP_IF_MISSING:
        if col in df.columns:
            df = df[df[col].notna()]

    return df.reset_index(drop=True)


def encode_procedure_side_value(val):
    """0.5 = Right/Left only, 1 = BOTH or Right+Left, else 0."""
    if pd.isna(val):
        return 0.0
    text = str(val).strip().upper()
    if not text or text == "NON":
        return 0.0
    if text == "BOTH":
        return 1.0

    parts = [p.strip() for p in text.split(",") if p.strip()]
    sides = set()
    for part in parts:
        if part == "BOTH":
            return 1.0
        if part in ("RIGHT", "LEFT"):
            sides.add(part)

    if "RIGHT" in sides and "LEFT" in sides:
        return 1.0
    if sides == {"RIGHT"} or sides == {"LEFT"}:
        return 0.5
    return 0.0


def encode_procedure_side(series):
    """Encode procedure side column to 0 / 0.5 / 1."""
    return series.apply(encode_procedure_side_value).astype("float64")


def drop_surgeries_timestamp_cols(df):
    """Remove all timestamp columns except surgery date."""
    cols_to_drop = [
        c for c in SURGERIES_RAW_TIMESTAMP_COLS + SURGERIES_COMPUTED_DATETIME_COLS
        if c in df.columns and c != SURGERY_DATE_COL
    ]
    return df.drop(columns=cols_to_drop)


def cast_surgeries_int_cols(df):
    """Convert key numeric columns to int."""
    df = df.copy()
    for col in SURGERIES_INT_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").round().astype("int64")
    return df


def clean_surgeries(df):
    """ניקוי טבלת ניתוחים."""
    df = df.copy()
    df.columns = df.columns.str.strip()

    # 1. הסרת עמודות
    cols_to_drop = [c for c in SURGERIES_COLS_TO_DROP if c in df.columns]
    df = df.drop(columns=cols_to_drop)

    # 2. מיפוי מחלקה מנתחת ל-3 מחלקות אב
    df["מחלקה מנתחת"] = map_parent_department(df["מחלקה מנתחת"])

    # 3. זמן כולל בחדר ניתוח (לפני קיבוץ)
    df = calculate_total_surgery_time(df)

    # 4. שורה אחת לכל ניתוח
    df = group_to_one_row_per_surgery(df, SURGERIES_GROUP_KEYS)
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")

    # 5. ערכים חסרים + הסרת שורות ללא target / גיל
    df = handle_surgeries_missing_values(df)

    # 5b. קידוד צד פרוצדורה: Right/Left=0.5, BOTH/שני צדדים=1, אחרת=0
    if PROCEDURE_SIDE_COL in df.columns:
        df[PROCEDURE_SIDE_COL] = encode_procedure_side(df[PROCEDURE_SIDE_COL])

    # 6. הסרת חותמות זמן (מלבד תאריך ניתוח)
    df = drop_surgeries_timestamp_cols(df)

    # 7. drop חדר + המרה ל-int
    extra_drop = [c for c in SURGERIES_COLS_TO_DROP_EXTRA if c in df.columns]
    df = df.drop(columns=extra_drop)

    df = cast_surgeries_int_cols(df)

    return df


def clean_bmi(df):
    """ניקוי טבלת BMI."""
    df = df.copy()
    df.columns = df.columns.str.strip()

    if "Patient" in df.columns:
        df = df.rename(columns={"Patient": "patient"})

    cols_to_drop = [c for c in BMI_COLS_TO_DROP if c in df.columns]
    df = df.drop(columns=cols_to_drop)

    df["מספר מקרה"] = pd.to_numeric(df["מספר מקרה"], errors="coerce")
    df["תאריך ניתוח"] = pd.to_datetime(df["תאריך ניתוח"], errors="coerce")

    # drop rows without merge keys
    df = df[df["מספר מקרה"].notna() & df["תאריך ניתוח"].notna()]

    # keep last row per (מספר מקרה, תאריך ניתוח)
    df = df.drop_duplicates(subset=BMI_GROUP_KEYS, keep="last")
    df["מספר מקרה"] = df["מספר מקרה"].round().astype("int64")

    return df.reset_index(drop=True)


def merge_surgeries_bmi(surgeries, bmi):
    """Left join BMI onto surgeries by (מספר מקרה, תאריך ניתוח)."""
    surgeries = surgeries.copy()
    bmi = bmi.copy()

    surgeries["תאריך ניתוח"] = pd.to_datetime(surgeries["תאריך ניתוח"], errors="coerce")

    bmi_cols = [c for c in bmi.columns if c not in BMI_GROUP_KEYS and c != "patient"]
    bmi_side = bmi[BMI_GROUP_KEYS + bmi_cols]

    return surgeries.merge(bmi_side, on=BMI_GROUP_KEYS, how="left")


def clean_smoking(df):
    """ניקוי טבלת עישון."""
    df = df.copy()
    df.columns = df.columns.str.strip()

    if "Patient" in df.columns:
        df = df.rename(columns={"Patient": "patient"})

    df["מספר מקרה"] = pd.to_numeric(df["מספר מקרה"], errors="coerce")
    df["מועד ניתוח"] = pd.to_datetime(df["מועד ניתוח"], errors="coerce").dt.normalize()

    # remove rows without case number
    df = df[df["מספר מקרה"].notna()]

    # keep last per (מספר מקרה, מועד ניתוח)
    df = df.drop_duplicates(subset=SMOKING_GROUP_KEYS, keep="last")
    df["מספר מקרה"] = df["מספר מקרה"].round().astype("int64")

    if SMOKING_COL in df.columns:
        df[SMOKING_COL] = df[SMOKING_COL].fillna(SMOKING_FILL_UNKNOWN)

    return df.reset_index(drop=True)


def merge_surgeries_smoking(merged, smoking):
    """Left join smoking onto surgeries anchor (תאריך ניתוח ↔ מועד ניתוח)."""
    merged = merged.copy()
    smoking = smoking.copy()

    merged[SURGERY_DATE_COL] = pd.to_datetime(merged[SURGERY_DATE_COL], errors="coerce").dt.normalize()
    smoking["מועד ניתוח"] = pd.to_datetime(smoking["מועד ניתוח"], errors="coerce").dt.normalize()

    smoking_cols = [
        c for c in smoking.columns
        if c not in SMOKING_GROUP_KEYS and c != "patient"
    ]
    smoking_side = smoking[SMOKING_GROUP_KEYS + smoking_cols]

    merged = merged.merge(
        smoking_side,
        left_on=["מספר מקרה", SURGERY_DATE_COL],
        right_on=["מספר מקרה", "מועד ניתוח"],
        how="left",
    )
    merged = merged.drop(columns=["מועד ניתוח"], errors="ignore")
    merged[SMOKING_COL] = merged[SMOKING_COL].fillna(SMOKING_FILL_UNKNOWN)

    return merged


def prepare_case_date_table(df, case_col_raw):
    """Rename case column, parse keys, drop rows without מספר מקרה."""
    df = df.copy()
    df.columns = df.columns.str.strip()

    if "Patient" in df.columns:
        df = df.rename(columns={"Patient": "patient"})

    if case_col_raw in df.columns and case_col_raw != "מספר מקרה":
        df = df.rename(columns={case_col_raw: "מספר מקרה"})

    df["מספר מקרה"] = pd.to_numeric(df["מספר מקרה"], errors="coerce")
    df["תאריך ניתוח"] = pd.to_datetime(df["תאריך ניתוח"], errors="coerce").dt.normalize()
    df = df[df["מספר מקרה"].notna() & df["תאריך ניתוח"].notna()]
    df["מספר מקרה"] = df["מספר מקרה"].round().astype("int64")
    return df.reset_index(drop=True)


def merge_case_date_table(merged, side, group_keys, exclude_cols=None):
    """Left join a case+date table onto the surgeries anchor."""
    merged = merged.copy()
    side = side.copy()
    exclude_cols = exclude_cols or [PATIENT_KEY]

    merged[SURGERY_DATE_COL] = pd.to_datetime(merged[SURGERY_DATE_COL], errors="coerce").dt.normalize()
    side["תאריך ניתוח"] = pd.to_datetime(side["תאריך ניתוח"], errors="coerce").dt.normalize()

    side_cols = [c for c in side.columns if c not in group_keys + exclude_cols]
    return merged.merge(side[group_keys + side_cols], on=group_keys, how="left")


def aggregate_values(df, group_key, value_col, output_col):
    """One row per key with comma-separated unique values."""
    return (
        df.groupby(group_key, as_index=False)[value_col]
        .agg(join_if_many)
        .rename(columns={value_col: output_col})
    )


def prepare_patient_table(df):
    """Rename Patient -> patient and drop rows without patient id."""
    df = df.copy()
    df.columns = df.columns.str.strip()
    if "Patient" in df.columns:
        df = df.rename(columns={"Patient": PATIENT_KEY})
    df[PATIENT_KEY] = pd.to_numeric(df[PATIENT_KEY], errors="coerce")
    df = df[df[PATIENT_KEY].notna()]
    df[PATIENT_KEY] = df[PATIENT_KEY].round().astype("int64")
    return df.reset_index(drop=True)


def merge_patient_table(merged, patient_side):
    """Left join a patient-level aggregated table."""
    return merged.merge(patient_side, on=PATIENT_KEY, how="left")


def clean_bp(df):
    """ניקוי טבלת לחץ דם."""
    df = prepare_case_date_table(df, BP_CASE_COL_RAW)
    df = df.drop_duplicates(subset=BP_GROUP_KEYS, keep="last")

    sbp = pd.to_numeric(df[BP_SBP_COL], errors="coerce")
    dbp = pd.to_numeric(df[BP_DBP_COL], errors="coerce")
    df.loc[(sbp <= 30) | (sbp >= 350), BP_SBP_COL] = np.nan
    df.loc[(dbp <= 20) | (dbp >= 250), BP_DBP_COL] = np.nan

    keep_cols = BP_GROUP_KEYS + [BP_SBP_COL, BP_DBP_COL]
    return df[keep_cols].reset_index(drop=True)


def clean_saturation(df):
    """ניקוי טבלת סטורציה."""
    df = prepare_case_date_table(df, SAT_CASE_COL_RAW)
    df = df.drop_duplicates(subset=SAT_GROUP_KEYS, keep="last")

    sat = pd.to_numeric(df[SAT_COL], errors="coerce")
    df.loc[(sat < 50) | (sat > 100), SAT_COL] = np.nan

    return df[SAT_GROUP_KEYS + [SAT_COL]].reset_index(drop=True)


def clean_background_diseases(df):
    """מחלות רקע — שורה אחת לכל patient עם רשימת ICD9."""
    df = prepare_patient_table(df)
    return aggregate_values(df, PATIENT_KEY, "ICD9", BG_OUTPUT_COL)


def clean_drug_allergy(df):
    """רגישות לתרופות — שורה אחת לכל patient."""
    df = prepare_patient_table(df)
    return aggregate_values(df, PATIENT_KEY, "Drug_Name", DRUG_ALLERGY_OUTPUT_COL)


def clean_other_allergy(df):
    """רגישות אחרת — שורה אחת לכל patient."""
    df = prepare_patient_table(df)
    return aggregate_values(df, PATIENT_KEY, "רגישות", OTHER_ALLERGY_OUTPUT_COL)


def clean_chronic_meds(df):
    """תרופות קבועות — שורה אחת לכל patient."""
    df = prepare_patient_table(df)
    return aggregate_values(df, PATIENT_KEY, "Drug_Name", CHRONIC_MEDS_OUTPUT_COL)


def clean_surgery_meds(df):
    """תרופות בניתוח — שורה אחת לכל מספר מקרה עם רשימת תרופות."""
    df = df.copy()
    df.columns = df.columns.str.strip()

    if "Patient" in df.columns:
        df = df.rename(columns={"Patient": PATIENT_KEY})

    cols_to_drop = [c for c in SURGERY_MEDS_DROP_COLS if c in df.columns]
    df = df.drop(columns=cols_to_drop)

    df[SURGERY_MEDS_KEY] = pd.to_numeric(df[SURGERY_MEDS_KEY], errors="coerce")
    df = df[df[SURGERY_MEDS_KEY].notna()]
    df = df.drop_duplicates(subset=[SURGERY_MEDS_KEY, SURGERY_MEDS_VALUE_COL], keep="last")
    df[SURGERY_MEDS_KEY] = df[SURGERY_MEDS_KEY].round().astype("int64")

    return aggregate_values(df, SURGERY_MEDS_KEY, SURGERY_MEDS_VALUE_COL, SURGERY_MEDS_OUTPUT_COL)


def merge_bp(merged, bp):
    return merge_case_date_table(merged, bp, BP_GROUP_KEYS)


def merge_saturation(merged, saturation):
    return merge_case_date_table(merged, saturation, SAT_GROUP_KEYS)


def merge_background_diseases(merged, background):
    return merge_patient_table(merged, background)


def merge_drug_allergy(merged, drug_allergy):
    return merge_patient_table(merged, drug_allergy)


def merge_other_allergy(merged, other_allergy):
    return merge_patient_table(merged, other_allergy)


def merge_chronic_meds(merged, chronic_meds):
    return merge_patient_table(merged, chronic_meds)


def merge_surgery_meds(merged, surgery_meds):
    return merged.merge(surgery_meds, on=SURGERY_MEDS_KEY, how="left")


def clean_labs(df):
    """ניקוי טבלת בדיקות מעבדה."""
    df = df.copy()
    if len(df.columns) == len(LAB_COLUMNS):
        df.columns = LAB_COLUMNS

    if "Patient" in df.columns:
        df = df.rename(columns={"Patient": "patient"})

    df["מספר מקרה"] = pd.to_numeric(df["מספר מקרה"], errors="coerce")
    df = df[df["מספר מקרה"].notna()]
    df[LAB_DATE_COL] = pd.to_datetime(df[LAB_DATE_COL], errors="coerce")
    df["מספר מקרה"] = df["מספר מקרה"].round().astype("int64")
    df[LAB_VALUE_COL] = pd.to_numeric(df[LAB_VALUE_COL], errors="coerce")
    df = df[df[LAB_TEST_COL].isin(LAB_SELECTED_TESTS)]
    return df.reset_index(drop=True)


def _pick_closest_lab_dates(surgery_keys, labs):
    """For each (מספר מקרה, תאריך ניתוח), pick closest תאריך בדיקה."""
    lab_dates = labs[["מספר מקרה", LAB_DATE_COL]].drop_duplicates()
    pairs = surgery_keys.merge(lab_dates, on="מספר מקרה", how="left")
    pairs["diff"] = (pairs[LAB_DATE_COL] - pairs[SURGERY_DATE_COL]).abs()
    closest = (
        pairs.sort_values("diff")
        .drop_duplicates(subset=["מספר מקרה", SURGERY_DATE_COL], keep="first")
    )
    return closest.dropna(subset=[LAB_DATE_COL])


def merge_labs(merged, labs):
    """Pivot selected lab tests wide; closest lab date per surgery row; missing=0."""
    merged = merged.copy()
    labs = labs.copy()

    merged[SURGERY_DATE_COL] = pd.to_datetime(
        merged[SURGERY_DATE_COL], errors="coerce"
    ).dt.normalize()

    surgery_keys = merged[["מספר מקרה", SURGERY_DATE_COL]].drop_duplicates()
    closest = _pick_closest_lab_dates(surgery_keys, labs)

    labs_match = labs.merge(
        closest,
        on=["מספר מקרה", LAB_DATE_COL],
        how="inner",
    )
    labs_match = labs_match.drop_duplicates(
        subset=["מספר מקרה", SURGERY_DATE_COL, LAB_TEST_COL],
        keep="last",
    )

    wide = labs_match.pivot(
        index=["מספר מקרה", SURGERY_DATE_COL],
        columns=LAB_TEST_COL,
        values=LAB_VALUE_COL,
    ).reset_index()
    wide.columns.name = None
    wide[SURGERY_DATE_COL] = pd.to_datetime(wide[SURGERY_DATE_COL]).dt.normalize()

    for col in LAB_SELECTED_TESTS:
        if col not in wide.columns:
            wide[col] = np.nan

    merged = merged.merge(
        wide[["מספר מקרה", SURGERY_DATE_COL] + LAB_SELECTED_TESTS],
        on=["מספר מקרה", SURGERY_DATE_COL],
        how="left",
    )

    for col in LAB_SELECTED_TESTS:
        merged[col] = pd.to_numeric(merged[col], errors="coerce").fillna(LABS_FILL_VALUE)

    n_with_labs = (merged[LAB_SELECTED_TESTS] != LABS_FILL_VALUE).any(axis=1).sum()
    print(f"lab columns: {len(LAB_SELECTED_TESTS)}")
    print(f"rows with lab data: {n_with_labs:,} / {len(merged):,}")

    return merged


def map_smoking_binary(series):
    """Map Smoking codes: 1=מעשן, 2/3/empty/UNKNOWN=לא מעשן."""
    numeric = pd.to_numeric(series, errors="coerce")
    result = pd.Series(NON_SMOKER_LABEL, index=series.index, dtype="string")
    result[numeric == 1] = SMOKER_LABEL
    return result


def split_combined_category(value):
    """Split A+B / A,B / A + B into separate category labels."""
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() in ("nan", "none", "<na>") or text.upper() == "UNKNOWN":
        return []
    text = re.sub(r"\s*[,+/|&]\s*", "+", text)
    text = re.sub(r"\s*\+\s*", "+", text)
    return [p.strip() for p in text.split("+") if p.strip()]


def standardize_anesthesia_type(value):
    """Unify combined anesthesia labels (A+B, A,B, ...) to A_B before OHE."""
    if pd.isna(value):
        return value
    text = str(value).strip()
    if not text or text.lower() in ("nan", "none", "<na>"):
        return text
    if text.upper() == "UNKNOWN":
        return text
    parts = split_combined_category(value)
    if len(parts) <= 1:
        return parts[0] if parts else text
    return "_".join(parts)


def encode_multilabel_ohe(df, col, prefix):
    """Multi-label OHE: A+B marks both prefix_A and prefix_B as 1."""
    df = df.copy()
    parts_series = df[col].apply(split_combined_category)
    all_parts = sorted({p for parts in parts_series for p in parts})
    if not all_parts:
        return df.drop(columns=[col])
    for part in all_parts:
        col_name = f"{prefix}_{part}"
        df[col_name] = parts_series.apply(lambda ps, p=part: int(p in ps))
    return df.drop(columns=[col])


def encode_merged_categoricals(df):
    """One-hot encode smoking, gender, and surgery categorical columns."""
    df = df.copy()

    if SMOKING_COL in df.columns:
        smoking_bin = map_smoking_binary(df[SMOKING_COL])
        smoking_dummies = pd.get_dummies(
            smoking_bin, prefix=SMOKING_OHE_PREFIX, dtype=int
        )
        df = df.drop(columns=[SMOKING_COL])
        df = pd.concat([df, smoking_dummies], axis=1)

    for col in OHE_CATEGORICAL_COLS:
        if col not in df.columns:
            continue
        if col == SURGERY_TYPE_COL:
            df = encode_multilabel_ohe(df, col, col)
            continue
        values = df[col].astype(str)
        if col == ANESTHESIA_TYPE_COL:
            values = values.apply(standardize_anesthesia_type)
        dummies = pd.get_dummies(values, prefix=col, dtype=int)
        df = df.drop(columns=[col])
        df = pd.concat([df, dummies], axis=1)

    drop_unknown = [c for c in OHE_DROP_UNKNOWN_COLS if c in df.columns]
    if drop_unknown:
        df = df.drop(columns=drop_unknown)

    return df


def count_comma_values(value):
    """Count non-empty comma-separated items."""
    if pd.isna(value):
        return 0
    parts = [p.strip() for p in str(value).split(",") if p.strip()]
    return len(parts)


DRUG_FORM_WORDS = {
    "TAB", "TABLET", "TABLETS", "CAPLET", "CAPLETS", "CAP", "CAPS", "CAPSULE", "CAPSULES",
    "INJ", "INJECTION", "INJECTIONS", "AMP", "AMPOULE", "AMPOULES", "SOLUTION", "SOLUTIONS",
    "SYRUP", "CREAM", "OINTMENT", "GEL", "PATCH", "SPRAY", "DROPS", "SUSPENSION", "POWDER",
    "GRANULES", "SUPPOSITORY", "OVULE", "OVULES", "PREFILLED", "SYRINGE", "VIAL", "VIALS",
    "INFUSION", "INOVAMED", "PRESERVATIVE", "FREE", "AND", "IN", "PLASTIC", "CONTAINER",
}

_DRUG_INDEX_CACHE = None


def load_drug_index(path=DRUG_INDEX_PATH):
    """Load canonical drug names; return set plus longest/shortest sort orders."""
    names = (
        pd.read_csv(path, dtype=str)["DrugName"]
        .dropna()
        .str.strip()
        .str.upper()
        .unique()
    )
    index_set = set(names)
    sorted_longest = sorted(names, key=len, reverse=True)
    sorted_shortest = sorted(names, key=len)
    return index_set, sorted_longest, sorted_shortest


def get_drug_index():
    """Load drug index once and cache for repeated normalization."""
    global _DRUG_INDEX_CACHE
    if _DRUG_INDEX_CACHE is None:
        _DRUG_INDEX_CACHE = load_drug_index()
    return _DRUG_INDEX_CACHE


def normalize_drug_for_matching(raw):
    """Uppercase and strip dosage, form, and route tokens for index matching."""
    s = str(raw).strip().upper()
    if not s or s in ("NAN", "NONE", "<NA>"):
        return ""
    s = re.sub(r"\s*\*\*.*?\*\*", "", s).strip()
    s = s.replace("-", " ").replace("'", "")
    s = re.sub(r"\s+", " ", s)
    s = re.sub(
        r"\s+\d[\d./\s%-]*(?:MG|ML|G|MCG|IU|UNITS?|µG|UG|%|MEQ|MCG/ML).*$",
        "",
        s,
        flags=re.IGNORECASE,
    )
    s = re.sub(r"\s+\d.*$", "", s)
    tokens = [t for t in s.split() if t not in DRUG_FORM_WORDS]
    return " ".join(tokens).strip()


def _drug_word_boundary_prefix(prefix, text):
    return text == prefix or text.startswith(prefix + " ")


def match_canonical_drug(raw, index_set, sorted_longest, sorted_shortest):
    """Map one raw drug string to a canonical FDA DrugName from the index."""
    s = normalize_drug_for_matching(raw)
    if not s:
        return None
    tokens = s.split()
    for n in range(len(tokens), 0, -1):
        candidate = " ".join(tokens[:n])
        if candidate in index_set:
            return candidate
    for canonical in sorted_longest:
        c = canonical.replace("'", "")
        if _drug_word_boundary_prefix(c, s):
            return canonical
    for canonical in sorted_shortest:
        c = canonical.replace("'", "")
        if _drug_word_boundary_prefix(s, c):
            return canonical
    return None


def clean_drugs_with_index(value, drug_index=None):
    """Normalize comma-separated drugs to unique canonical names from the index."""
    if pd.isna(value):
        return pd.NA
    if drug_index is None:
        drug_index = get_drug_index()
    index_set, sorted_longest, sorted_shortest = drug_index
    found = []
    for part in str(value).split(","):
        canonical = match_canonical_drug(
            part, index_set, sorted_longest, sorted_shortest
        )
        name_to_add = canonical if canonical else normalize_drug_for_matching(part)
        if name_to_add and name_to_add not in found:
            found.append(name_to_add)
    return ",".join(found) if found else pd.NA


def add_age_group_flags(df):
    """One-hot flags: Age Group_Young / Middle-aged / Elderly."""
    df = df.copy()
    age = pd.to_numeric(df["גיל"], errors="coerce")
    df["Age Group_Young"] = ((age >= 0) & (age <= AGE_YOUNG_MAX)).astype(int)
    df["Age Group_Middle-aged"] = ((age > AGE_YOUNG_MAX) & (age < AGE_ELDERLY_MIN)).astype(int)
    df["Age Group_Elderly"] = (age >= AGE_ELDERLY_MIN).astype(int)
    return df


def add_bmi_category_flags(df):
    """One-hot flags for BMI categories."""
    df = df.copy()
    bmi = pd.to_numeric(df["BMI"], errors="coerce")
    df["BMI Category_Underweight"] = (bmi < BMI_UNDERWEIGHT_MAX).astype(int)
    df["BMI Category_Normal"] = ((bmi >= BMI_UNDERWEIGHT_MAX) & (bmi < BMI_NORMAL_MAX)).astype(int)
    df["BMI Category_Overweight"] = ((bmi >= BMI_NORMAL_MAX) & (bmi < BMI_OVERWEIGHT_MAX)).astype(int)
    df["BMI Category_Obese"] = (bmi >= BMI_OVERWEIGHT_MAX).astype(int)
    return df


def engineer_merged_features(df):
    """Drop unused cols; add age/BMI flags, drug cleaning, and counters."""
    df = df.copy()

    drop_cols = [c for c in COLS_TO_DROP if c in df.columns]
    df = df.drop(columns=drop_cols)

    df = add_age_group_flags(df)
    df = add_bmi_category_flags(df)

    if SURGERY_MEDS_OUTPUT_COL in df.columns or DRUG_ALLERGY_OUTPUT_COL in df.columns:
        drug_index = get_drug_index()
    if SURGERY_MEDS_OUTPUT_COL in df.columns:
        df[CLEANED_DRUGS_COL] = df[SURGERY_MEDS_OUTPUT_COL].apply(
            clean_drugs_with_index, drug_index=drug_index
        )
    if DRUG_ALLERGY_OUTPUT_COL in df.columns:
        df[CLEANED_DRUG_ALLERGIES_COL] = df[DRUG_ALLERGY_OUTPUT_COL].apply(
            clean_drugs_with_index, drug_index=drug_index
        )

    if BG_OUTPUT_COL in df.columns:
        df[BG_COUNT_COL] = df[BG_OUTPUT_COL].apply(count_comma_values)
    if CLEANED_DRUG_ALLERGIES_COL in df.columns:
        df[DRUG_ALLERGY_COUNT_COL] = df[CLEANED_DRUG_ALLERGIES_COL].apply(count_comma_values)
    elif DRUG_ALLERGY_OUTPUT_COL in df.columns:
        df[DRUG_ALLERGY_COUNT_COL] = df[DRUG_ALLERGY_OUTPUT_COL].apply(count_comma_values)
    if "קוד פרוצדורה" in df.columns:
        df[PROCEDURE_CODE_COUNT_COL] = df["קוד פרוצדורה"].apply(count_comma_values)

    return df


def zscore_column(series, treat_zero_as_missing=False):
    """Z-score normalization; optional: treat 0 as missing for mean/std."""
    s = pd.to_numeric(series, errors="coerce")
    stats = s.replace(0, np.nan) if treat_zero_as_missing else s
    mu = stats.mean()
    sigma = stats.std(ddof=0)
    if pd.isna(sigma) or sigma == 0:
        z = pd.Series(0.0, index=s.index)
    else:
        z = (s - mu) / sigma
    return z.fillna(0.0)


def normalize_ohe_category_suffix(suffix):
    """Standardize OHE category suffix (merge + , / | & variants)."""
    text = str(suffix).strip()
    text = re.sub(r"[,/|&]", " + ", text)
    text = re.sub(r"\s*\+\s*", " + ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if " + " in text:
        parts = sorted(p.strip() for p in text.split(" + ") if p.strip())
        return " + ".join(parts)
    return text


def explode_combined_ohe_columns(df, prefix):
    """Split legacy combined OHE columns (A + B) into atomic prefix_A, prefix_B."""
    df = df.copy()
    prefix_tag = f"{prefix}_"
    combined_cols = [
        c for c in df.columns
        if c.startswith(prefix_tag) and len(split_combined_category(c[len(prefix_tag):])) > 1
    ]
    for col in combined_cols:
        parts = split_combined_category(col[len(prefix_tag):])
        vals = df[col]
        for part in parts:
            atomic = f"{prefix_tag}{part}"
            if atomic not in df.columns:
                df[atomic] = 0
            df[atomic] = df[atomic].combine(vals, max).astype(int)
        df = df.drop(columns=[col])
    return df


def merge_ohe_prefix_columns(df, prefix):
    """Merge duplicate one-hot columns that differ only by separators."""
    df = df.copy()
    prefix_tag = f"{prefix}_"
    cols = [c for c in df.columns if c.startswith(prefix_tag)]
    if not cols:
        return df

    groups = {}
    for col in cols:
        suffix = col[len(prefix_tag):]
        canonical = prefix_tag + normalize_ohe_category_suffix(suffix)
        groups.setdefault(canonical, []).append(col)

    for canonical, members in groups.items():
        if len(members) == 1 and members[0] == canonical:
            continue
        merged_vals = df[members].max(axis=1).astype(int)
        drop_cols = [c for c in members if c != canonical]
        df[canonical] = merged_vals
        df = df.drop(columns=drop_cols)

    return df


def consolidate_smoker_column(df):
    """Convert עישון_מעשן / עישון_לא מעשן into single smoker column."""
    df = df.copy()
    smoker_col = f"{SMOKING_OHE_PREFIX}_{SMOKER_LABEL}"
    non_smoker_col = f"{SMOKING_OHE_PREFIX}_{NON_SMOKER_LABEL}"
    if smoker_col not in df.columns:
        return df

    df[SMOKER_COL] = df[smoker_col].astype(int)
    drop_cols = [c for c in [smoker_col, non_smoker_col] if c in df.columns]
    return df.drop(columns=drop_cols)


def prepare_final_dataset(df):
    """Normalization and feature cleanup for ML-ready dataset."""
    df = df.copy()

    if "BMI" in df.columns:
        df[BMI_Z_COL] = zscore_column(df["BMI"])

    if SAT_COL in df.columns:
        sat = pd.to_numeric(df[SAT_COL], errors="coerce")
        df[SAT_NORM_COL] = (sat / 100.0).fillna(0.0)

    for col in LAB_SELECTED_TESTS:
        if col in df.columns:
            df[f"{col}_Z"] = zscore_column(df[col], treat_zero_as_missing=True)

    lab_raw_drop = [c for c in LAB_SELECTED_TESTS if c in df.columns]
    if lab_raw_drop:
        df = df.drop(columns=lab_raw_drop)

    if BP_DBP_COL in df.columns:
        df[DBP_Z_COL] = zscore_column(df[BP_DBP_COL])
    if BP_SBP_COL in df.columns:
        df[SBP_Z_COL] = zscore_column(df[BP_SBP_COL])

    df = explode_combined_ohe_columns(df, SURGERY_TYPE_OHE_PREFIX)
    df = merge_ohe_prefix_columns(df, ANESTHESIA_OHE_PREFIX)
    df = consolidate_smoker_column(df)

    return df


def _safe_icd9_code(code):
    """Sanitize ICD9 code for use in a column name."""
    return re.sub(r"[^\w.]", "_", str(code).strip())


def encode_top_k_icd9_ohe(df, col=BG_OUTPUT_COL, k=BG_TOP_K_ICD9):
    """One-hot encode the top-K most frequent ICD9 codes from background diseases."""
    df = df.copy()
    if col not in df.columns:
        return df, []

    from collections import Counter

    counter = Counter()
    row_codes = {}
    for idx, val in df[col].items():
        if pd.isna(val):
            row_codes[idx] = []
            continue
        codes = [c.strip() for c in str(val).split(",") if c.strip()]
        row_codes[idx] = codes
        counter.update(codes)

    top_k_codes = [code for code, _ in counter.most_common(k)]
    for code in top_k_codes:
        ohe_col = f"{BG_ICD9_OHE_PREFIX}_{_safe_icd9_code(code)}"
        flags = pd.Series(
            [int(code in row_codes[idx]) for idx in df.index],
            index=df.index,
            dtype=int,
        )
        df[ohe_col] = flags

    return df, top_k_codes


def _build_column_rename_map():
    """Hebrew / internal names → English ML column names."""
    surgery_type_map = {
        f"{SURGERY_TYPE_OHE_PREFIX}_אלקטיבי": "ST_Elective",
        f"{SURGERY_TYPE_OHE_PREFIX}_ססיה": "ST_Sesia",
        f"{SURGERY_TYPE_OHE_PREFIX}_דחוף": "ST_Urgent",
        f"{SURGERY_TYPE_OHE_PREFIX}_בהול": "ST_Emergency",
        f"{SURGERY_TYPE_OHE_PREFIX}_פעולה לא ניתוחית": "ST_Non_Surgical_Operation",
        f"{SURGERY_TYPE_OHE_PREFIX}_קציר איברים": "ST_Organ_Procurement",
    }
    anesthesia_map = {
        f"{ANESTHESIA_OHE_PREFIX}_General": "AT_General",
        f"{ANESTHESIA_OHE_PREFIX}_General_Local": "AT_General_Local",
        f"{ANESTHESIA_OHE_PREFIX}_General_Regional": "AT_General_Regional",
        f"{ANESTHESIA_OHE_PREFIX}_Local": "AT_Local",
        f"{ANESTHESIA_OHE_PREFIX}_Local_Regional": "AT_Local_Regional",
        f"{ANESTHESIA_OHE_PREFIX}_Local_Sedation": "AT_Local_Sedation",
        f"{ANESTHESIA_OHE_PREFIX}_Other": "AT_Other",
        f"{ANESTHESIA_OHE_PREFIX}_Regional": "AT_Regional",
        f"{ANESTHESIA_OHE_PREFIX}_Regional_Sedation": "AT_Regional_Sedation",
        f"{ANESTHESIA_OHE_PREFIX}_Sedation": "AT_Sedation",
        f"{ANESTHESIA_OHE_PREFIX}_Spinal": "AT_Spinal",
        f"{ANESTHESIA_OHE_PREFIX}_Spinal_Epidural": "AT_Spinal_Epidural",
        f"{ANESTHESIA_OHE_PREFIX}_Epidural": "AT_Epidural",
    }
    weekday_map = {
        "יום ניתוח_ראשון": "Sunday",
        "יום ניתוח_שני": "Monday",
        "יום ניתוח_שלישי": "Tuesday",
        "יום ניתוח_רביעי": "Wednesday",
        "יום ניתוח_חמישי": "Thursday",
        "יום ניתוח_שישי": "Friday",
        "יום ניתוח_שבת": "Saturday",
    }
    rename = {
        PATIENT_KEY: "Patient",
        "Medical Record": "Medical_Record",
        "מספר מקרה": "Case_Number",
        "מחלקה מנתחת": "Surgical_Department",
        SURGERY_DATE_COL: "Surgery_Date",
        "גיל": "Age",
        "מין_זכר": "Male",
        "מין_נקבה": "Female",
        "BMI": "BMI",
        BMI_Z_COL: "Normalized_BMI",
        "קוד פרוצדורה": "Procedure_Code",
        "צד פרוצדורה": "Procedure_Side",
        SAT_NORM_COL: "Oxygen_Saturation",
        SBP_Z_COL: "BP_Systolic_Before_Surgery",
        DBP_Z_COL: "BP_Diastolic_Before_Surgery",
        SMOKER_COL: "Smoking",
        "סוג מקרה_אשפוז": "CT_Hospitalized",
        "סוג מקרה_אמבולטורי": "CT_Ambulatory",
        SURGERY_MEDS_OUTPUT_COL: "Drug_Names",
        DRUG_ALLERGY_OUTPUT_COL: "Allergy_Drug_Names",
        BG_OUTPUT_COL: "Disease_ICD9_Codes",
        TARGET_COL: "Total_Surgery_Time",
        ESTIMATED_SURGERY_TIME_COL: ESTIMATED_SURGERY_TIME_COL,
        CLEANED_DRUGS_COL: "Cleaned_Drug_Names",
        CLEANED_DRUG_ALLERGIES_COL: "Cleaned_Allergy_Drug_Names",
        NUM_MEDICATIONS_COL: NUM_MEDICATIONS_COL,
        NUM_ALLERGY_MEDICATIONS_COL: NUM_ALLERGY_MEDICATIONS_COL,
        SURGERY_TIME_ANOMALY_COL: SURGERY_TIME_ANOMALY_COL,
        "Age Group_Young": "Age_Group_Young",
        "Age Group_Middle-aged": "Age_Group_Middle-aged",
        "Age Group_Elderly": "Age_Group_Elderly",
        "BMI Category_Underweight": "BMI_Category_Underweight",
        "BMI Category_Normal": "BMI_Category_Normal",
        "BMI Category_Overweight": "BMI_Category_Overweight",
        "BMI Category_Obese": "BMI_Category_Obese",
    }
    rename.update(surgery_type_map)
    rename.update(anesthesia_map)
    rename.update(weekday_map)
    return rename


def _build_final_column_order(top_k_icd9_codes):
    """Return ordered English column list for the ML-ready dataset."""
    icd9_cols = [f"{BG_ICD9_OHE_PREFIX}_{_safe_icd9_code(c)}" for c in top_k_icd9_codes]
    lab_cols = [f"{col}_Z" for col in LAB_SELECTED_TESTS]
    return [
        "Patient",
        "Medical_Record",
        "Case_Number",
        "Surgical_Department",
        "Surgery_Date",
        "Age",
        "Male",
        "Female",
        "BMI",
        "Normalized_BMI",
        "Procedure_Code",
        "Procedure_Side",
        "Oxygen_Saturation",
        "BP_Systolic_Before_Surgery",
        "BP_Diastolic_Before_Surgery",
        "Smoking",
        "CT_Hospitalized",
        "CT_Ambulatory",
        "ST_Elective",
        "ST_Elective_Sesia",
        "ST_Emergency",
        "ST_Non_Surgical_Operation",
        "ST_Organ_Procurement",
        "ST_Sesia",
        "ST_Urgent",
        "AT_General",
        "AT_General_Local",
        "AT_General_Regional",
        "AT_Local",
        "AT_Local_Regional",
        "AT_Other",
        "AT_Regional",
        "AT_Sedation",
        "AT_Spinal",
        "AT_Spinal_Epidural",
        "AT_Epidural",
        "AT_Local_Sedation",
        "AT_Regional_Sedation",
        "Sunday",
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Drug_Names",
        "Allergy_Drug_Names",
        "Disease_ICD9_Codes",
        "Total_Surgery_Time",
        ESTIMATED_SURGERY_TIME_COL,
        "Cleaned_Drug_Names",
        "Cleaned_Allergy_Drug_Names",
        NUM_MEDICATIONS_COL,
        NUM_ALLERGY_MEDICATIONS_COL,
        SURGERY_TIME_ANOMALY_COL,
        "Age_Group_Young",
        "Age_Group_Middle-aged",
        "Age_Group_Elderly",
        "BMI_Category_Underweight",
        "BMI_Category_Normal",
        "BMI_Category_Overweight",
        "BMI_Category_Obese",
        *icd9_cols,
        *lab_cols,
    ]


def finalize_english_ml_dataset(df, top_k_icd9=BG_TOP_K_ICD9):
    """Translate columns to English, add ICD9 OHE, drop extras, and reorder."""
    df = df.copy()

    if "duration_quality" in df.columns:
        df[SURGERY_TIME_ANOMALY_COL] = (
            df["duration_quality"] != "valid"
        ).astype(int)

    if CLEANED_DRUGS_COL in df.columns:
        df[NUM_MEDICATIONS_COL] = df[CLEANED_DRUGS_COL].apply(count_comma_values)
    if CLEANED_DRUG_ALLERGIES_COL in df.columns:
        df[NUM_ALLERGY_MEDICATIONS_COL] = df[CLEANED_DRUG_ALLERGIES_COL].apply(
            count_comma_values
        )
    elif DRUG_ALLERGY_COUNT_COL in df.columns:
        df[NUM_ALLERGY_MEDICATIONS_COL] = df[DRUG_ALLERGY_COUNT_COL]

    if DRUG_ALLERGY_COUNT_COL in df.columns:
        df = df.drop(columns=[DRUG_ALLERGY_COUNT_COL])

    df[ESTIMATED_SURGERY_TIME_COL] = np.nan

    df, top_k_codes = encode_top_k_icd9_ohe(df, k=top_k_icd9)

    drop_cols = [c for c in ML_COLUMNS_DROP if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    rename_map = _build_column_rename_map()
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    ordered = _build_final_column_order(top_k_codes)
    ordered_existing = [c for c in ordered if c in df.columns]
    extra_cols = [c for c in df.columns if c not in ordered_existing]
    df = df[ordered_existing + extra_cols]

    return df


def export_ml_dataset_by_department(df, export_dir=ML_EXPORT_DIR):
    """Split final ML dataset by surgical department and export CSV per department."""
    export_dir = Path(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)

    if SURGICAL_DEPARTMENT_COL not in df.columns:
        raise KeyError(f"Missing column: {SURGICAL_DEPARTMENT_COL}")

    exported = {}
    for dept, filename in DEPARTMENT_EXPORT_FILES.items():
        subset = df[df[SURGICAL_DEPARTMENT_COL] == dept]
        path = export_dir / filename
        subset.to_csv(path, index=False)
        exported[dept] = path
        print(f"{dept}: {len(subset):,} rows -> {path.resolve()}")

    missing = sorted(
        set(df[SURGICAL_DEPARTMENT_COL].dropna().unique())
        - set(DEPARTMENT_EXPORT_FILES.keys())
    )
    if missing:
        print(f"warning: unmapped departments (not exported): {missing}")

    return exported


def invalid_height(row):
    age = row["גיל"]
    h = row["גובה"]
    if pd.isna(age) or pd.isna(h):
        return False
    if 1 <= age <= 4:
        return (h < 48.0) or (h > 120.0)
    if 5 <= age <= 12:
        return (h < 80.0) or (h > 200.0)
    if 13 <= age <= 17:
        return (h < 100.0) or (h > 220.0)
    return (h < 100.0) or (h > 220.0)


def invalid_weight(row):
    w = row["משקל"]
    if pd.isna(w):
        return False
    return (w < 20.0) or (w > 300.0)


def invalid_bmi_value(bmi):
    if pd.isna(bmi):
        return False
    return (bmi < 10.0) or (bmi > 80.0)


def _fill_by_age_gender_neighborhood(df, value_col, max_k=10):
    """Fill one column using same-gender mean by age, widening ±1..max_k years."""
    valid_ages = df["גיל"].dropna()
    if valid_ages.empty:
        return df

    min_age = int(valid_ages.min())
    max_age = int(valid_ages.max())

    age_gender_mean = (
        df.groupby(["_Male", "גיל"], as_index=False)[value_col]
        .mean()
        .rename(columns={value_col: "mean_age"})
    )

    full_grid = pd.MultiIndex.from_product(
        [[0, 1], range(min_age, max_age + 1)],
        names=["_Male", "גיל"],
    ).to_frame(index=False)

    lookup = full_grid.merge(age_gender_mean, on=["_Male", "גיל"], how="left")
    ages = lookup["גיל"].to_numpy()
    males = lookup["_Male"].to_numpy()
    mean_vals = lookup["mean_age"].to_numpy()
    knn_vals = np.full_like(mean_vals, np.nan, dtype=float)

    for g in (0, 1):
        idxs = np.where(males == g)[0]
        ages_g = ages[idxs]
        means_g = mean_vals[idxs]

        for i_local, age0 in enumerate(ages_g):
            if not np.isnan(means_g[i_local]):
                knn_vals[idxs[i_local]] = means_g[i_local]
                continue
            for k in range(1, max_k + 1):
                in_window = (ages_g >= age0 - k) & (ages_g <= age0 + k)
                candidates = means_g[in_window]
                candidates = candidates[~np.isnan(candidates)]
                if candidates.size > 0:
                    knn_vals[idxs[i_local]] = candidates.mean()
                    break

    lookup["knn_mean"] = knn_vals
    df = df.merge(
        lookup[["_Male", "גיל", "mean_age", "knn_mean"]],
        on=["_Male", "גיל"],
        how="left",
    )
    df[value_col] = df[value_col].fillna(df["mean_age"]).fillna(df["knn_mean"])
    df = df.drop(columns=["mean_age", "knn_mean"])
    return df


def impute_merged_bmi_metrics(df, max_k=BMI_IMPUTE_MAX_AGE_K):
    """Invalidate outliers, impute height/weight by age+gender, recalc BMI."""
    df = df.copy()

    for col in ["גובה", "משקל", "BMI"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["גיל"] = pd.to_numeric(df["גיל"], errors="coerce")
    df["_Male"] = df["מין"].map({"זכר": 1, "נקבה": 0})

    invalid_h = df.apply(invalid_height, axis=1)
    invalid_w = df.apply(invalid_weight, axis=1)
    invalid_bmi = df["BMI"].apply(invalid_bmi_value)

    # rows that need BMI recalc after height/weight fix
    needs_bmi_recalc = (
        df["BMI"].isna() | invalid_bmi
        | invalid_h | invalid_w
        | df["גובה"].isna() | df["משקל"].isna()
    )

    df.loc[invalid_h, "גובה"] = np.nan
    df.loc[invalid_w, "משקל"] = np.nan
    df.loc[invalid_bmi, "BMI"] = np.nan

    df = _fill_by_age_gender_neighborhood(df, "גובה", max_k=max_k)
    df = _fill_by_age_gender_neighborhood(df, "משקל", max_k=max_k)

    recalc_mask = needs_bmi_recalc & df["גובה"].notna() & df["משקל"].notna()
    df.loc[recalc_mask, "BMI"] = (
        df.loc[recalc_mask, "משקל"] / (df.loc[recalc_mask, "גובה"] / 100) ** 2
    )

    df = df.drop(columns=["_Male"])
    return df

---
## Block 4 — load.py

שמירת טבלאות מעובדות.

In [4]:
def save_staged(df, table_name):
    """Save cleaned table to data/clean/staged/."""
    path = STAGED_DIR / f"{table_name}.pkl"
    df.to_pickle(path)
    print(f"saved {table_name}: {df.shape} -> {path}")
    return path

---
## Block 5 — run.py

Orchestrator: extract → clean → save.  
בשרת: `run_surgeries(conn=my_connection)`.

In [5]:
def run_surgeries(conn=None, save=True):
    """Full ETL for surgeries table."""
    df = extract_surgeries(conn)
    print(f"extract: {df.shape}")

    df = clean_surgeries(df)
    print(f"clean:   {df.shape}")

    if save:
        save_staged(df, "surgeries")

    return df


def run_bmi(conn=None, save=True):
    """Full ETL for BMI table."""
    df = extract_bmi(conn)
    print(f"extract: {df.shape}")

    df = clean_bmi(df)
    print(f"clean:   {df.shape}")

    if save:
        save_staged(df, "bmi")

    return df


def run_merge_surgeries_bmi(surgeries=None, bmi=None, save=True):
    """Merge cleaned BMI into surgeries anchor table."""
    if surgeries is None:
        surgeries = pd.read_pickle(STAGED_DIR / "surgeries.pkl")
    if bmi is None:
        bmi = pd.read_pickle(STAGED_DIR / "bmi.pkl")

    merged = merge_surgeries_bmi(surgeries, bmi)
    print(f"merged:  {merged.shape}")
    print(f"BMI matched: {merged['BMI'].notna().sum():,} / {len(merged):,}")

    merged = impute_merged_bmi_metrics(merged)
    print(f"after impute — height nulls: {merged['גובה'].isna().sum():,}")
    print(f"after impute — weight nulls: {merged['משקל'].isna().sum():,}")
    print(f"after impute — BMI nulls:    {merged['BMI'].isna().sum():,}")

    if save:
        save_staged(merged, MERGED_SURGERIES_NAME)

    return merged


def run_smoking(conn=None, save=True):
    """Full ETL for smoking table."""
    df = extract_smoking(conn)
    print(f"extract: {df.shape}")

    df = clean_smoking(df)
    print(f"clean:   {df.shape}")

    if save:
        save_staged(df, "smoking")

    return df


def run_merge_surgeries_smoking(merged=None, smoking=None, save=True):
    """Merge cleaned smoking into surgeries+BMI table."""
    if merged is None:
        merged = pd.read_pickle(STAGED_DIR / MERGED_SURGERIES_NAME)
    if smoking is None:
        smoking = pd.read_pickle(STAGED_DIR / "smoking.pkl")

    merged = merge_surgeries_smoking(merged, smoking)
    print(f"merged:  {merged.shape}")
    n_matched = (merged[SMOKING_COL] != SMOKING_FILL_UNKNOWN).sum()
    print(f"Smoking matched: {n_matched:,} / {len(merged):,}")

    merged = encode_merged_categoricals(merged)
    print(f"after OHE: {merged.shape}")

    if save:
        save_staged(merged, MERGED_AFTER_SMOKING_NAME)

    return merged


def _run_clean_table(extract_fn, clean_fn, table_name, conn=None, save=True):
    df = extract_fn(conn)
    print(f"extract: {df.shape}")
    df = clean_fn(df)
    print(f"clean:   {df.shape}")
    if save:
        save_staged(df, table_name)
    return df


def run_bp(conn=None, save=True):
    return _run_clean_table(extract_bp, clean_bp, "bp", conn, save)


def run_saturation(conn=None, save=True):
    return _run_clean_table(extract_saturation, clean_saturation, "saturation", conn, save)


def run_background_diseases(conn=None, save=True):
    return _run_clean_table(extract_background_diseases, clean_background_diseases, "background_diseases", conn, save)


def run_drug_allergy(conn=None, save=True):
    return _run_clean_table(extract_drug_allergy, clean_drug_allergy, "drug_allergy", conn, save)


def run_other_allergy(conn=None, save=True):
    return _run_clean_table(extract_other_allergy, clean_other_allergy, "other_allergy", conn, save)


def run_chronic_meds(conn=None, save=True):
    return _run_clean_table(extract_chronic_meds, clean_chronic_meds, "chronic_meds", conn, save)


def run_surgery_meds(conn=None, save=True):
    return _run_clean_table(extract_surgery_meds, clean_surgery_meds, "surgery_meds", conn, save)


def run_merge_bp(merged=None, bp=None, save=True):
    if merged is None:
        merged = pd.read_pickle(STAGED_DIR / MERGED_AFTER_SMOKING_NAME)
    if bp is None:
        bp = pd.read_pickle(STAGED_DIR / "bp.pkl")
    merged = merge_bp(merged, bp)
    print(f"merged: {merged.shape}")
    print(f"BP matched: {merged[BP_SBP_COL].notna().sum():,} / {len(merged):,}")
    if save:
        save_staged(merged, MERGED_AFTER_BP_NAME)
    return merged


def run_merge_saturation(merged=None, saturation=None, save=True):
    if merged is None:
        merged = pd.read_pickle(STAGED_DIR / MERGED_AFTER_BP_NAME)
    if saturation is None:
        saturation = pd.read_pickle(STAGED_DIR / "saturation.pkl")
    merged = merge_saturation(merged, saturation)
    print(f"merged: {merged.shape}")
    print(f"saturation matched: {merged[SAT_COL].notna().sum():,} / {len(merged):,}")
    if save:
        save_staged(merged, MERGED_AFTER_SAT_NAME)
    return merged


def run_merge_patient_tables(
    merged=None,
    background=None,
    drug_allergy=None,
    save=True,
):
    if merged is None:
        merged = pd.read_pickle(STAGED_DIR / MERGED_AFTER_SAT_NAME)
    if background is None:
        background = pd.read_pickle(STAGED_DIR / "background_diseases.pkl")
    if drug_allergy is None:
        drug_allergy = pd.read_pickle(STAGED_DIR / "drug_allergy.pkl")

    merged = merge_background_diseases(merged, background)
    merged = merge_drug_allergy(merged, drug_allergy)

    print(f"merged: {merged.shape}")
    print(f"{BG_OUTPUT_COL} matched: {merged[BG_OUTPUT_COL].notna().sum():,} / {len(merged):,}")
    print(f"{DRUG_ALLERGY_OUTPUT_COL} matched: {merged[DRUG_ALLERGY_OUTPUT_COL].notna().sum():,} / {len(merged):,}")

    if save:
        save_staged(merged, MERGED_AFTER_PATIENT_NAME)
    return merged


def run_merge_surgery_meds(merged=None, surgery_meds=None, save=True):
    if merged is None:
        merged = pd.read_pickle(STAGED_DIR / MERGED_AFTER_PATIENT_NAME)
    if surgery_meds is None:
        surgery_meds = pd.read_pickle(STAGED_DIR / "surgery_meds.pkl")

    merged = merge_surgery_meds(merged, surgery_meds)
    merged = engineer_merged_features(merged)
    print(f"merged: {merged.shape}")
    print(f"{SURGERY_MEDS_OUTPUT_COL} matched: {merged[SURGERY_MEDS_OUTPUT_COL].notna().sum():,} / {len(merged):,}")

    if save:
        save_staged(merged, MERGED_AFTER_SURGERY_MEDS_NAME)
    return merged


def run_labs(conn=None, save=True):
    return _run_clean_table(extract_labs, clean_labs, "labs", conn, save)


def run_merge_labs(merged=None, labs=None, save=True):
    if merged is None:
        merged = pd.read_pickle(STAGED_DIR / MERGED_AFTER_SURGERY_MEDS_NAME)
    if labs is None:
        labs = pd.read_pickle(STAGED_DIR / "labs.pkl")

    merged = merge_labs(merged, labs)
    merged = prepare_final_dataset(merged)
    merged = finalize_english_ml_dataset(merged)
    print(f"merged: {merged.shape}")

    if save:
        save_staged(merged, MERGED_AFTER_LABS_NAME)
        save_staged(merged, ML_DATASET_NAME)
    return merged

---
## Block 6 — profile (notebook only)

בדיקות לפני/אחרי ניקוי. **לא** עובר ל-production pipeline.

In [6]:
def profile(df, name, key_cols=None):
    """Print basic stats about a table. Notebook helper only."""
    print(f"--- {name} ---")
    print(f"rows: {len(df)}, cols: {df.shape[1]}")

    if key_cols:
        for col in key_cols:
            if col not in df.columns:
                continue
            n_unique = df[col].nunique()
            n_dups = df.duplicated(subset=[col]).sum()
            print(f"  {col}: {n_unique} unique, {n_dups} duplicate rows")

    print("  null % per column:")
    for col in df.columns:
        pct = df[col].isna().mean() * 100
        if pct > 0:
            print(f"    {col}: {pct:.1f}%")

---
# Table 1 — ניתוחים

טבלת **עוגן**. כל שאר הטבלאות יצטרפו אליה בשלב המיזוג.

### החלטות ניקוי

| # | פעולה | סטטוס |
|---|--------|--------|
| 1 | Drop 11 עמודות (F45, F46, הרדמה פירוט, צוות, הערות…) | ✅ |
| 2 | שורה אחת ל-**(מספר מקרה, Medical Record)** | ✅ |
| 3 | עמודות מרובות בקבוצה → רשימה מופרדת ב-`,` ; ערך יחיד → נשמר כערך | ✅ |
| 4 | Map `מחלקה מנתחת` → 3 מחלקות אב (אורטופדיה / כירורגיה / אף אוזן גרון) | ✅ |
| 5 | `total_surgery_time` — OR occupancy with consistent start/end + quality flags | ✅ |
| 6 | Drop rows missing `total_surgery_time` / `גיל` | ✅ |
| 7 | Fill UNKNOWN / NON for categorical nulls | ✅ |
| 8 | `צד פרוצדורה`: Right/Left=0.5, BOTH/שני צדדים=1, אחרת=0 | ✅ |
| 9 | Drop `חדר` | ✅ |
| 10 | Drop all timestamp cols except `תאריך ניתוח` | ✅ |
| 10 | `int` dtype: `מספר מקרה`, `גיל`, `total_surgery_time` | ✅ |
| 10 | בדיקת outliers בגיל | ⏸ |
| 11 | סינון תאריכים עתידיים | ⏸ |

### 1.1 Extract + Profile

In [7]:
# extract only (no save yet — cleaning not done)
surgeries_raw = extract_surgeries()
profile(surgeries_raw, "ניתוחים — raw", key_cols=["מספר מקרה", "patient"])
surgeries_raw.head(2)

--- ניתוחים — raw ---
rows: 46909, cols: 44
  מספר מקרה: 33500 unique, 13408 duplicate rows
  patient: 28461 unique, 18448 duplicate rows
  null % per column:
    מספר מקרה: 0.0%
    גיל: 0.0%
    מחלקה מנתחת נוספת: 99.5%
    חדר: 9.2%
    קוד פרוצדורה: 6.1%
    שם פרוצדורה: 6.1%
    הערה לפרוצדורה: 77.8%
    צד פרוצדורה: 54.4%
    קוד אבחנה: 13.1%
    שם אבחנה: 13.1%
    הערה לאבחנה: 76.6%
    תאריך כניסה לחדר ניתוח: 0.9%
    שעת כניסה לחדר ניתוח: 0.9%
    תאריך תחילת הרדמה: 1.9%
    שעת תחילת הרדמה: 1.9%
    תאריך סיום הרדמה: 8.4%
    שעת סיום הרדמה: 8.4%
    תאריך סיום ניתוח: 0.6%
    שעת סיום ניתוח: 0.6%
    תאריך יציאה מחדר ניתוח: 8.8%
    שעת יציאה מחדר ניתוח: 8.8%
    תאריך שחרור מח ניתוח: 0.0%
    שעת שחרור מח ניתוח: 0.0%
    סוג ניתוח: 1.2%
    פעולה ניתוחית מיילדותית: 100.0%
    סוג הרדמה: 0.9%
    הרדמה פירוט: 98.9%
    מנתח אחראי: 3.1%
    מנתח ראשי: 38.9%
     עוזר מנתח: 65.8%
     אחות רחוצה: 8.6%
     אחות מסתובבת: 6.7%
     מרדים: 14.1%
    F45: 100.0%
    F46: 100.0%


,מספר מקרה,Medical Record,גיל,מין,סוג מקרה,תאריך ניתוח,יום ניתוח,מחלקה מנתחת,מחלקה מנתחת נוספת,חדר,קוד פרוצדורה,שם פרוצדורה,הערה לפרוצדורה,צד פרוצדורה,קוד אבחנה,שם אבחנה,הערה לאבחנה,תאריך כניסה לחדר ניתוח,שעת כניסה לחדר ניתוח,תאריך תחילת ניתוח,שעת תחילת ניתוח,תאריך תחילת הרדמה,שעת תחילת הרדמה,תאריך סיום הרדמה,שעת סיום הרדמה,תאריך סיום ניתוח,שעת סיום ניתוח,תאריך יציאה מחדר ניתוח,שעת יציאה מחדר ניתוח,תאריך שחרור מח ניתוח,שעת שחרור מח ניתוח,סוג ניתוח,פעולה ניתוחית מיילדותית,סוג הרדמה,הרדמה פירוט,מנתח אחראי,מנתח ראשי,עוזר מנתח,אחות רחוצה,אחות מסתובבת,מרדים,F45,F46,patient
0,11642461.0,2084001,22.0,זכר,אשפוז,2023-07-03,שני,אף אוזן גרון,NaN,חדר 7,21.69.05,PARTIAL TURBINECTOMY,NaN,NaN,470,DEVIATED NASAL SEPTUM,NASAL POLYPS?,2023-07-03,11:06:00,2023-07-03,11:32:00,2023-07-03,11:10:00,2023-07-03,12:50:00,2023-07-03,12:42:00,2023-07-03,12:54:00,2023-07-03,12:55:00,אלקטיבי,NaN,General,NaN,דר' מעיין דור,דר' חג'וג' מג'ד,NaN,אוחיון דלנה,אברמוביץ' יפה,דר' גבאס אחמד,NaN,NaN,447553
1,11642461.0,2084001,22.0,זכר,אשפוז,2023-07-03,שני,אף אוזן גרון,NaN,חדר 7,21.88,SEPTOPLASTY NEC,NaN,NaN,478,HYPERTROPHY OF NASAL TURBINATES,NaN,2023-07-03,11:06:00,2023-07-03,11:32:00,2023-07-03,11:10:00,2023-07-03,12:50:00,2023-07-03,12:42:00,2023-07-03,12:54:00,2023-07-03,12:55:00,אלקטיבי,NaN,General,NaN,דר' מעיין דור,דר' חג'וג' מג'ד,NaN,אוחיון דלנה,אברמוביץ' יפה,דר' גבאס אחמד,NaN,NaN,447553


In [8]:
df = surgeries_raw

n_cases = df["מספר מקרה"].nunique()
print(f"rows:          {len(df)}")
print(f"unique cases:  {n_cases}")
print(f"rows / case:   {len(df) / n_cases:.2f}")

multi = df.groupby("מספר מקרה").size()
print(f"cases with >1 row: {(multi > 1).sum()}")

print(f"\nage > 100: {(df['גיל'] > 100).sum()}")
print(df["גיל"].describe())

dates = pd.to_datetime(df["תאריך ניתוח"], errors="coerce")
print(f"\ndate min: {dates.min()}")
print(f"date max: {dates.max()}")
print(f"future dates: {(dates > pd.Timestamp.today()).sum()}")

rows:          46909
unique cases:  33500
rows / case:   1.40
cases with >1 row: 8717

age > 100: 58
count    46899.000000
mean        50.839783
std         22.982989
min          1.000000
25%         32.000000
50%         52.000000
75%         70.000000
max        111.000000
Name: גיל, dtype: float64

date min: 2020-01-20 00:00:00
date max: 2026-02-18 00:00:00
future dates: 0


### 1.2 Clean + validate


In [9]:
surgeries = run_surgeries(save=False)
profile(surgeries, "ניתוחים — clean", key_cols=["מספר מקרה", "Medical Record"])

print(f"\nrows: {len(surgeries):,}  (was 46,909 raw procedure rows)")
print(f"unique cases: {surgeries['מספר מקרה'].nunique():,}")
print(f"cases with >1 Medical Record: {surgeries.duplicated('מספר מקרה', keep=False).sum():,}")

multi_proc = surgeries["שם פרוצדורה"].astype(str).str.contains(",", na=False).sum()
print(f"rows with multiple procedures: {multi_proc:,}")

print(f"\nparent departments:\n{surgeries['מחלקה מנתחת'].value_counts()}")
print(f"\n{TARGET_COL} (minutes):\n{surgeries[TARGET_COL].describe()}")
print(f"\nduration_quality:\n{surgeries['duration_quality'].value_counts()}")
print(f"\nstart_time_source:\n{surgeries['start_time_source'].value_counts()}")
print(f"\nend_time_source:\n{surgeries['end_time_source'].value_counts()}")

# דוגמה: ניתוח עם כמה פרוצדורות
example = surgeries[surgeries["שם פרוצדורה"].astype(str).str.contains(",", na=False)].head(1)
if len(example):
    print("\nexample multi-procedure row:")
    print(example[["מספר מקרה", "Medical Record", "שם פרוצדורה", "גיל"]].to_string(index=False))

surgeries.head(3)

extract: (46909, 44)
clean:   (34359, 20)
--- ניתוחים — clean ---
rows: 34359, cols: 20
  מספר מקרה: 33342 unique, 1017 duplicate rows
  Medical Record: 34359 unique, 0 duplicate rows
  null % per column:

rows: 34,359  (was 46,909 raw procedure rows)
unique cases: 33,342
cases with >1 Medical Record: 1,790
rows with multiple procedures: 8,444

parent departments:
מחלקה מנתחת
אורטופדיה       14772
כירורגיה        14258
אף אוזן גרון     5329
Name: count, dtype: int64

total_surgery_time (minutes):
count    34359.000000
mean       100.402020
std         69.214363
min          1.000000
25%         53.000000
50%         83.000000
75%        130.000000
max        669.000000
Name: total_surgery_time, dtype: float64

duration_quality:
duration_quality
valid    34359
Name: count, dtype: int64

start_time_source:
start_time_source
entry_to_or         34068
anesthesia_start      276
surgery_start          15
Name: count, dtype: int64

end_time_source:
end_time_source
discharge_close_to_end    34

,מספר מקרה,Medical Record,גיל,מין,סוג מקרה,תאריך ניתוח,יום ניתוח,מחלקה מנתחת,קוד פרוצדורה,שם פרוצדורה,צד פרוצדורה,קוד אבחנה,שם אבחנה,סוג ניתוח,סוג הרדמה,patient,start_time_source,end_time_source,duration_quality,total_surgery_time
0,4093,1338976,55,זכר,אמבולטורי,2022-03-27,ראשון,כירורגיה,"41.5,54.11,55.4","TOTAL SPLENECTOMY,EXPLORATORY LAPAROTOMY,PARTI...",0.5,"54.11,860.3,866.1,866.12,805.4,865.1,812","EXPLORATORY LAPAROTOMY,TRAUMATIC HEMOTHORAX WI...",בהול,General,443795,entry_to_or,discharge_close_to_end,valid,81
1,4093,1341489,55,זכר,אמבולטורי,2022-03-29,שלישי,כירורגיה,"54.11,54.61","EXPLORATORY LAPAROTOMY,RECLOSURE OF POSTOPERAT...",0.0,"866.1,866.12,879.2,865.1",UNSPECIFIED INJURY TO KIDNEY WITH OPEN WOUND I...,דחוף,General,443795,entry_to_or,discharge_close_to_end,valid,92
2,4093,1346096,55,זכר,אמבולטורי,2022-04-01,שישי,אורטופדיה,"86.22,93.53","DEBRIDEMENT OF WOUND,INFECTION,OR BURN),APPLIC...",0.5,823.1,OPEN FRACTURE OF UPPER END OF TIBIA,דחוף,Local + Regional,443795,entry_to_or,discharge_close_to_end,valid,62


In [10]:
# בדיקה של מקרי המקסימום
# top20 = surgeries.nlargest(20, "total_surgery_time")
# surgeries.head(25)

In [11]:
surgeries.shape

(34359, 20)

In [12]:
eda_summary = pd.DataFrame({
    "column": surgeries.columns,
    "dtype": surgeries.dtypes.astype(str),
    "missing_count": surgeries.isnull().sum(),
    "missing_pct": (surgeries.isnull().mean() * 100).round(2),
    "n_unique": surgeries.nunique(dropna=True)
})

display(eda_summary.sort_values("missing_pct", ascending=False))

,column,dtype,missing_count,missing_pct,n_unique
מספר מקרה,מספר מקרה,int64,0,0.0,33342
Medical Record,Medical Record,int64,0,0.0,34359
גיל,גיל,int64,0,0.0,108
מין,מין,object,0,0.0,2
סוג מקרה,סוג מקרה,object,0,0.0,2
תאריך ניתוח,תאריך ניתוח,datetime64[ns],0,0.0,2184
יום ניתוח,יום ניתוח,object,0,0.0,7
מחלקה מנתחת,מחלקה מנתחת,object,0,0.0,3
קוד פרוצדורה,קוד פרוצדורה,object,0,0.0,4973
שם פרוצדורה,שם פרוצדורה,object,0,0.0,5619


### 1.3 Save


In [13]:
# surgeries = run_surgeries(save=True)

---
# Table 2 — BMI

### החלטות ניקוי

| # | פעולה | סטטוס |
|---|--------|--------|
| 1 | Drop `דם` | ✅ |
| 2 | Dedup `(מספר מקרה, תאריך ניתוח)` — keep last | ✅ |
| 3 | Left join onto surgeries | ✅ |

### 2.1 Extract + Clean

In [14]:
bmi = run_bmi(save=False)
profile(bmi, "BMI — clean", key_cols=BMI_GROUP_KEYS)
print(f"\nduplicates after clean: {bmi.duplicated(BMI_GROUP_KEYS).sum()}")
bmi.head(3)

extract: (34521, 7)
clean:   (34498, 6)
--- BMI — clean ---
rows: 34498, cols: 6
  מספר מקרה: 33500 unique, 998 duplicate rows
  תאריך ניתוח: 2184 unique, 32314 duplicate rows
  null % per column:
    גובה: 20.9%
    משקל: 5.4%
    BMI: 21.0%

duplicates after clean: 0


,patient,תאריך ניתוח,מספר מקרה,גובה,משקל,BMI
0,100343,2020-05-24,11071228,182.0,89.0,26.868735
1,100349,2025-09-28,11976373,NaN,95.0,NaN
2,100352,2020-05-19,11070099,175.0,73.0,23.836734


### 2.2 Merge with surgeries

In [15]:
# use in-memory tables, or load from staged/
# surgeries = pd.read_pickle(STAGED_DIR / "surgeries.pkl")

merged = merge_surgeries_bmi(surgeries, bmi)
merged = impute_merged_bmi_metrics(merged)
print(f"surgeries: {surgeries.shape} -> merged: {merged.shape}")
print(f"height nulls: {merged['גובה'].isna().sum():,}")
print(f"weight nulls: {merged['משקל'].isna().sum():,}")
print(f"BMI nulls:    {merged['BMI'].isna().sum():,}")
print(f"new cols: {[c for c in merged.columns if c not in surgeries.columns]}")
merged.head(3)

surgeries: (34359, 20) -> merged: (34359, 23)
height nulls: 0
weight nulls: 0
BMI nulls:    0
new cols: ['גובה', 'משקל', 'BMI']


,מספר מקרה,Medical Record,גיל,מין,סוג מקרה,תאריך ניתוח,יום ניתוח,מחלקה מנתחת,קוד פרוצדורה,שם פרוצדורה,צד פרוצדורה,קוד אבחנה,שם אבחנה,סוג ניתוח,סוג הרדמה,patient,start_time_source,end_time_source,duration_quality,total_surgery_time,גובה,משקל,BMI
0,4093,1338976,55,זכר,אמבולטורי,2022-03-27,ראשון,כירורגיה,"41.5,54.11,55.4","TOTAL SPLENECTOMY,EXPLORATORY LAPAROTOMY,PARTI...",0.5,"54.11,860.3,866.1,866.12,805.4,865.1,812","EXPLORATORY LAPAROTOMY,TRAUMATIC HEMOTHORAX WI...",בהול,General,443795,entry_to_or,discharge_close_to_end,valid,81,180.0,95.0,29.320987
1,4093,1341489,55,זכר,אמבולטורי,2022-03-29,שלישי,כירורגיה,"54.11,54.61","EXPLORATORY LAPAROTOMY,RECLOSURE OF POSTOPERAT...",0.0,"866.1,866.12,879.2,865.1",UNSPECIFIED INJURY TO KIDNEY WITH OPEN WOUND I...,דחוף,General,443795,entry_to_or,discharge_close_to_end,valid,92,180.0,95.0,29.320987
2,4093,1346096,55,זכר,אמבולטורי,2022-04-01,שישי,אורטופדיה,"86.22,93.53","DEBRIDEMENT OF WOUND,INFECTION,OR BURN),APPLIC...",0.5,823.1,OPEN FRACTURE OF UPPER END OF TIBIA,דחוף,Local + Regional,443795,entry_to_or,discharge_close_to_end,valid,62,180.0,95.0,29.320987


In [16]:
# merged.head()
merged.nlargest(20, "משקל")[
    ["patient", "מספר מקרה", "משקל", "גובה", "BMI"]
]

,patient,מספר מקרה,משקל,גובה,BMI
6760,357320,11191303,197.0,180.000000,60.802469
23634,382493,11810904,190.0,190.000000,52.631578
11800,382493,11477291,185.0,190.000000,51.246537
14050,495145,11546956,175.0,148.000000,79.894083
25334,118244,11861797,175.0,170.000000,60.553633
26827,241629,11904489,175.0,195.000000,46.022353
10225,434036,11435455,173.0,196.000000,45.033319
17587,259729,11644793,173.0,180.000000,53.395061
17588,259729,11644793,173.0,180.000000,53.395061
17589,259729,11644793,173.0,180.000000,53.395061


### 2.3 Save

In [17]:
# bmi = run_bmi(save=True)
# merged = run_merge_surgeries_bmi(surgeries=surgeries, bmi=bmi, save=True)

---
# Table 3 — עישון

### החלטות ניקוי

| # | פעולה | סטטוס |
|---|--------|--------|
| 1 | Drop שורות ללא `מספר מקרה` | ✅ |
| 2 | Dedup `(מספר מקרה, מועד ניתוח)` — keep last | ✅ |
| 3 | `Smoking` חסר → `UNKNOWN` (לפני מיזוג) | ✅ |
| 4 | Left join onto `surgeries_with_bmi` (`תאריך ניתוח` ↔ `מועד ניתוח`) | ✅ |
| 5 | OHE: עישון → `עישון_מעשן` / `עישון_לא מעשן` (1=מעשן, 2/3/ריק=לא מעשן) | ✅ |
| 6 | OHE: `מין`, `סוג מקרה`, `יום ניתוח`, `סוג ניתוח`, `סוג הרדמה` | ✅ |
| 7 | לפני OHE — סטנדרטיזציה של `סוג הרדמה` משולב (`A+B` → `A_B`) | ✅ |
| 8 | לפני OHE — `סוג ניתוח` משולב (`A+B`) מסומן בשני עמודות נפרדות | ✅ |
| 9 | אחרי OHE — drop `סוג ניתוח_UNKNOWN`, `סוג הרדמה_UNKNOWN` | ✅ |

### 3.1 Extract + Clean

In [18]:
smoking = run_smoking(save=False)
profile(smoking, "עישון — clean", key_cols=SMOKING_GROUP_KEYS)
print(f"duplicates after clean: {smoking.duplicated(subset=SMOKING_GROUP_KEYS).sum()}")
print(f"Smoking value counts:\n{smoking[SMOKING_COL].value_counts(dropna=False)}")
smoking.head()

extract: (31661, 4)
clean:   (30636, 4)
--- עישון — clean ---
rows: 30636, cols: 4
  מספר מקרה: 29659 unique, 977 duplicate rows
  מועד ניתוח: 2184 unique, 28452 duplicate rows
  null % per column:
duplicates after clean: 0
Smoking value counts:
Smoking
2.0        20832
1.0         9150
3.0          483
UNKNOWN      171
Name: count, dtype: int64


,patient,מספר מקרה,מועד ניתוח,Smoking
0,348295,11432846,2022-02-08,2.0
1,701394,11876662,2025-02-09,3.0
2,61493,11770783,2024-05-20,2.0
3,298529,11409054,2021-12-22,1.0
4,229077,31776481,2022-08-22,2.0


### 3.2 Merge with surgeries+BMI

In [19]:
# use in-memory tables, or load from staged/
# merged = pd.read_pickle(STAGED_DIR / MERGED_SURGERIES_NAME)

merged_with_smoking = merge_surgeries_smoking(merged, smoking)
merged_with_smoking = encode_merged_categoricals(merged_with_smoking)
print(f"merged: {merged.shape} -> {merged_with_smoking.shape}")
new_cols = [c for c in merged_with_smoking.columns if c not in merged.columns]
print(f"OHE cols ({len(new_cols)}): {new_cols[:10]}{'...' if len(new_cols) > 10 else ''}")
print(f"עישון counts:\n{merged_with_smoking.filter(like='עישון_').sum()}")
merged_with_smoking.filter(like="עישון_").head()

merged: (34359, 23) -> (34359, 50)
OHE cols (32): ['עישון_לא מעשן', 'עישון_מעשן', 'מין_זכר', 'מין_נקבה', 'סוג מקרה_אמבולטורי', 'סוג מקרה_אשפוז', 'יום ניתוח_חמישי', 'יום ניתוח_ראשון', 'יום ניתוח_רביעי', 'יום ניתוח_שבת']...
עישון counts:
עישון_לא מעשן    25244
עישון_מעשן        9115
dtype: int64


,עישון_לא מעשן,עישון_מעשן
0,1,0
1,1,0
2,1,0
3,1,0
4,1,0


In [20]:
merged_with_smoking.head()

,מספר מקרה,Medical Record,גיל,תאריך ניתוח,מחלקה מנתחת,קוד פרוצדורה,שם פרוצדורה,צד פרוצדורה,קוד אבחנה,שם אבחנה,patient,start_time_source,end_time_source,duration_quality,total_surgery_time,גובה,משקל,BMI,עישון_לא מעשן,עישון_מעשן,מין_זכר,מין_נקבה,סוג מקרה_אמבולטורי,סוג מקרה_אשפוז,יום ניתוח_חמישי,יום ניתוח_ראשון,יום ניתוח_רביעי,יום ניתוח_שבת,יום ניתוח_שישי,יום ניתוח_שלישי,יום ניתוח_שני,סוג ניתוח_אלקטיבי,סוג ניתוח_בהול,סוג ניתוח_דחוף,סוג ניתוח_ססיה,סוג ניתוח_פעולה לא ניתוחית,סוג ניתוח_קציר איברים,סוג הרדמה_Epidural,סוג הרדמה_General,סוג הרדמה_General_Local,סוג הרדמה_General_Regional,סוג הרדמה_Local,סוג הרדמה_Local_Regional,סוג הרדמה_Local_Sedation,סוג הרדמה_Other,סוג הרדמה_Regional,סוג הרדמה_Regional_Sedation,סוג הרדמה_Sedation,סוג הרדמה_Spinal,סוג הרדמה_Spinal_Epidural
0,4093,1338976,55,2022-03-27,כירורגיה,"41.5,54.11,55.4","TOTAL SPLENECTOMY,EXPLORATORY LAPAROTOMY,PARTI...",0.5,"54.11,860.3,866.1,866.12,805.4,865.1,812","EXPLORATORY LAPAROTOMY,TRAUMATIC HEMOTHORAX WI...",443795,entry_to_or,discharge_close_to_end,valid,81,180.000000,95.0,29.320987,1,0,1,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
1,4093,1341489,55,2022-03-29,כירורגיה,"54.11,54.61","EXPLORATORY LAPAROTOMY,RECLOSURE OF POSTOPERAT...",0.0,"866.1,866.12,879.2,865.1",UNSPECIFIED INJURY TO KIDNEY WITH OPEN WOUND I...,443795,entry_to_or,discharge_close_to_end,valid,92,180.000000,95.0,29.320987,1,0,1,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
2,4093,1346096,55,2022-04-01,אורטופדיה,"86.22,93.53","DEBRIDEMENT OF WOUND,INFECTION,OR BURN),APPLIC...",0.5,823.1,OPEN FRACTURE OF UPPER END OF TIBIA,443795,entry_to_or,discharge_close_to_end,valid,62,180.000000,95.0,29.320987,1,0,1,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
3,11015426,384447,40,2020-02-11,אורטופדיה,78.68,REMOVAL OF IMPLANTED DEVICE FROM TARSAL AND ME...,0.5,998.59,OTHER POSTOPERATIVE INFECTION,2162,entry_to_or,discharge_close_to_end,valid,193,163.923567,70.0,26.050451,1,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
4,11028836,347153,73,2020-01-21,אורטופדיה,81.59,REVISION OF JOINT REPLACEMENT OF LOWER EXTREMI...,0.5,996.44,PERI-PROSTHETIC FRACTURE AROUND PROSTHETIC JOINT,51923,entry_to_or,discharge_close_to_end,valid,284,176.000000,96.0,30.991735,1,0,0,1,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0


### 3.3 Save

In [21]:
# smoking = run_smoking(save=True)
# merged_with_smoking = run_merge_surgeries_smoking(merged=merged, smoking=smoking, save=True)

---
# Tables 4–11 — לחץ דם, סטורציה, מחלות רקע, רגישויות, תרופות

### 4. לחץ דם + 5. סטורציה

| טבלה | ניקוי | מיזוג |
|------|--------|--------|
| לחץ דם | drop ללא `מספר מקרה`; SBP≤30/SBP≥350→NaN; DBP≤20/DBP≥250→NaN; dedup | `(מספר מקרה, תאריך ניתוח)` |
| סטורציה | drop ללא `מספר מקרה`; <50 או >100→NaN; dedup | `(מספר מקרה, תאריך ניתוח)` |

### 6–8. טבלאות patient + תרופות בניתוח

מיזוג לפי `patient` — ערכים מרובים מצטברים לעמודה אחת מופרדת בפסיק.

| טבלה | עמודה מצטברת |
|------|----------------|
| מחלות רקע | `מחלות_רקע` (ICD9) + `מחלות_רקע_count` |
| רגישות לתרופות | `רגישות_לתרופות` + `CLEANED_DRUG_ALLERGIES` + count |

### תרופות בניתוח

drop `אופן מתן`; dedup keep last; צבירה → `תרופות_בניתוח` + `CLEANED_DRUGS` (נרמול מול `data/Drugs/unique_drug_names.csv`)

### Feature engineering (אחרי כל המיזוגים)

- drop: `חדר`, `רגישות_אחרת`, `תרופות_קבועות`
- Age Group flags: Young (≤39), Middle-aged (40–64), Elderly (≥65)
- BMI Category flags: Underweight / Normal / Overweight / Obese
- `CLEANED_DRUGS` — נרמול תרופות בניתוח מול אינדקס FDA
- `CLEANED_DRUG_ALLERGIES` — נרמול רגישות לתרופות (אותה לוגיקה)
- `קוד_פרוצדורה_count`

In [22]:
bp = run_bp(save=False)
saturation = run_saturation(save=False)
profile(bp, "לחץ דם — clean", key_cols=BP_GROUP_KEYS)
profile(saturation, "סטורציה — clean", key_cols=SAT_GROUP_KEYS)
bp.head(3)

extract: (35678, 5)
clean:   (32813, 4)
extract: (27625, 4)
clean:   (27605, 3)
--- לחץ דם — clean ---
rows: 32813, cols: 4
  מספר מקרה: 31816 unique, 997 duplicate rows
  תאריך ניתוח: 2184 unique, 30629 duplicate rows
  null % per column:
    היסטולי לפני ניתוח: 0.1%
    דיאסטולי לפני ניתוח: 0.1%
--- סטורציה — clean ---
rows: 27605, cols: 3
  מספר מקרה: 26615 unique, 990 duplicate rows
  תאריך ניתוח: 2182 unique, 25423 duplicate rows
  null % per column:
    סטורציה לפני ניתוח: 0.2%


,מספר מקרה,תאריך ניתוח,היסטולי לפני ניתוח,דיאסטולי לפני ניתוח
0,11028836,2020-01-21,116.0,62.0
1,11033725,2020-01-21,90.0,50.0
2,11035095,2020-01-20,140.0,70.0


In [23]:
background = run_background_diseases(save=False)
drug_allergy = run_drug_allergy(save=False)
surgery_meds = run_surgery_meds(save=False)

print("background:", background.shape)
print("drug_allergy:", drug_allergy.shape)
print("surgery_meds:", surgery_meds.shape)
background.head(3)

extract: (35359, 3)
clean:   (10781, 2)
extract: (8820, 3)
clean:   (5196, 2)
extract: (319950, 4)
clean:   (31791, 2)
background: (10781, 2)
drug_allergy: (5196, 2)
surgery_meds: (31791, 2)


,patient,מחלות_רקע
0,220,"401.9,V58.61,79.35,250.9,427.31,733,V13.85,821..."
1,226,"V15.82,401.9,435.2,564.1,272.2,434.9,726.13"
2,244,"696.8,293.84,493.9"


In [24]:
# merge onto smoking result (in-memory or staged)
# merged_with_smoking = pd.read_pickle(STAGED_DIR / MERGED_AFTER_SMOKING_NAME)

merged_with_smoking = merge_bp(merged_with_smoking, bp)
merged_with_smoking = merge_saturation(merged_with_smoking, saturation)
merged_with_smoking = merge_background_diseases(merged_with_smoking, background)
merged_with_smoking = merge_drug_allergy(merged_with_smoking, drug_allergy)
merged_with_smoking = merge_surgery_meds(merged_with_smoking, surgery_meds)
merged_with_smoking = engineer_merged_features(merged_with_smoking)

print(f"final shape: {merged_with_smoking.shape}")
print(f"BP matched: {merged_with_smoking[BP_SBP_COL].notna().sum():,}")
print(f"saturation matched: {merged_with_smoking[SAT_COL].notna().sum():,}")
print(f"{BG_OUTPUT_COL} matched: {merged_with_smoking[BG_OUTPUT_COL].notna().sum():,}")
print(f"{SURGERY_MEDS_OUTPUT_COL} matched: {merged_with_smoking[SURGERY_MEDS_OUTPUT_COL].notna().sum():,}")
merged_with_smoking[
    [CLEANED_DRUGS_COL, SURGERY_MEDS_OUTPUT_COL, BG_COUNT_COL, DRUG_ALLERGY_COUNT_COL,
     PROCEDURE_CODE_COUNT_COL, "Age Group_Young", "BMI Category_Normal"]
].head(3)

cleaned_drugs_export = merged_with_smoking[[CLEANED_DRUGS_COL, SURGERY_MEDS_OUTPUT_COL]].copy()
cleaned_drugs_export.to_csv(CLEANED_DRUGS_EXPORT_PATH, index=False)
print(f"Exported {len(cleaned_drugs_export):,} rows -> {CLEANED_DRUGS_EXPORT_PATH.resolve()}")

final shape: (34359, 68)
BP matched: 32,655
saturation matched: 27,441
מחלות_רקע matched: 14,189
תרופות_בניתוח matched: 32,668
Exported 34,359 rows -> /home/gur/Documents/Ruppin/CSY3/Final Project/סמסטר ב/ml_data_pipeline/data/Drugs/Output/cleaned_drugs_comparison.csv


In [25]:
merged_with_smoking.head()

,מספר מקרה,Medical Record,גיל,תאריך ניתוח,מחלקה מנתחת,קוד פרוצדורה,שם פרוצדורה,צד פרוצדורה,קוד אבחנה,שם אבחנה,patient,start_time_source,end_time_source,duration_quality,total_surgery_time,גובה,משקל,BMI,עישון_לא מעשן,עישון_מעשן,מין_זכר,מין_נקבה,סוג מקרה_אמבולטורי,סוג מקרה_אשפוז,יום ניתוח_חמישי,יום ניתוח_ראשון,יום ניתוח_רביעי,יום ניתוח_שבת,יום ניתוח_שישי,יום ניתוח_שלישי,יום ניתוח_שני,סוג ניתוח_אלקטיבי,סוג ניתוח_בהול,סוג ניתוח_דחוף,סוג ניתוח_ססיה,סוג ניתוח_פעולה לא ניתוחית,סוג ניתוח_קציר איברים,סוג הרדמה_Epidural,סוג הרדמה_General,סוג הרדמה_General_Local,סוג הרדמה_General_Regional,סוג הרדמה_Local,סוג הרדמה_Local_Regional,סוג הרדמה_Local_Sedation,סוג הרדמה_Other,סוג הרדמה_Regional,סוג הרדמה_Regional_Sedation,סוג הרדמה_Sedation,סוג הרדמה_Spinal,סוג הרדמה_Spinal_Epidural,היסטולי לפני ניתוח,דיאסטולי לפני ניתוח,סטורציה לפני ניתוח,מחלות_רקע,רגישות_לתרופות,תרופות_בניתוח,Age Group_Young,Age Group_Middle-aged,Age Group_Elderly,BMI Category_Underweight,BMI Category_Normal,BMI Category_Overweight,BMI Category_Obese,CLEANED_DRUGS,CLEANED_DRUG_ALLERGIES,מחלות_רקע_count,רגישות_לתרופות_count,קוד_פרוצדורה_count
0,4093,1338976,55,2022-03-27,כירורגיה,"41.5,54.11,55.4","TOTAL SPLENECTOMY,EXPLORATORY LAPAROTOMY,PARTI...",0.5,"54.11,860.3,866.1,866.12,805.4,865.1,812","EXPLORATORY LAPAROTOMY,TRAUMATIC HEMOTHORAX WI...",443795,entry_to_or,discharge_close_to_end,valid,81,180.000000,95.0,29.320987,1,0,1,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,140.0,65.0,100.0,250,NaN,"FRESH FROZEN PLASMA **BLOOD TRANSFUSION**,PROP...",0,1,0,0,0,1,0,"FRESH FROZEN PLASMA,PROPOFOL,MARCAINE,LACTATED...",<NA>,1,0,3
1,4093,1341489,55,2022-03-29,כירורגיה,"54.11,54.61","EXPLORATORY LAPAROTOMY,RECLOSURE OF POSTOPERAT...",0.0,"866.1,866.12,879.2,865.1",UNSPECIFIED INJURY TO KIDNEY WITH OPEN WOUND I...,443795,entry_to_or,discharge_close_to_end,valid,92,180.000000,95.0,29.320987,1,0,1,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,140.0,65.0,100.0,250,NaN,"FRESH FROZEN PLASMA **BLOOD TRANSFUSION**,PROP...",0,1,0,0,0,1,0,"FRESH FROZEN PLASMA,PROPOFOL,MARCAINE,LACTATED...",<NA>,1,0,2
2,4093,1346096,55,2022-04-01,אורטופדיה,"86.22,93.53","DEBRIDEMENT OF WOUND,INFECTION,OR BURN),APPLIC...",0.5,823.1,OPEN FRACTURE OF UPPER END OF TIBIA,443795,entry_to_or,discharge_close_to_end,valid,62,180.000000,95.0,29.320987,1,0,1,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,140.0,65.0,100.0,250,NaN,"FRESH FROZEN PLASMA **BLOOD TRANSFUSION**,PROP...",0,1,0,0,0,1,0,"FRESH FROZEN PLASMA,PROPOFOL,MARCAINE,LACTATED...",<NA>,1,0,2
3,11015426,384447,40,2020-02-11,אורטופדיה,78.68,REMOVAL OF IMPLANTED DEVICE FROM TARSAL AND ME...,0.5,998.59,OTHER POSTOPERATIVE INFECTION,2162,entry_to_or,discharge_close_to_end,valid,193,163.923567,70.0,26.050451,1,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,119.0,71.0,100.0,"V45.86,838.01,493.9,285.9,81,816,E957.9,825.21...","CEFORAL CAP 500MG,FOLEX 400 TAB","FENTANYL INJ 0.1 mg/2mL,EPHEDRINE SULFATE 29/M...",0,1,0,0,0,1,0,"FENTANYL,EPHEDRINE SULFATE,SODIUM CHLORIDE,LAC...","CEFORAL,FOLEX",15,2,1
4,11028836,347153,73,2020-01-21,אורטופדיה,81.59,REVISION OF JOINT REPLACEMENT OF LOWER EXTREMI...,0.5,996.44,PERI-PROSTHETIC FRACTURE AROUND PROSTHETIC JOINT,51923,entry_to_or,discharge_close_to_end,valid,284,176.000000,96.0,30.991735,1,0,0,1,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,116.0,62.0,95.0,"V15.82,F209,401.9","PENICILLIN,SULFA","ESMERON INJ 50MG/5ML,PARACETAMOL 1G INJECTION,...",0,0,1,0,0,0,1,"ESMERON,PARACETAMOL,PROPOFOL,MORPHINE HCL,EPHE...","PENICILLIN,SULFA",3,2,1


### Save tables 4–11

In [26]:
# run_bp(save=True); run_saturation(save=True)
# run_background_diseases(save=True); run_drug_allergy(save=True)
# run_other_allergy(save=True); run_chronic_meds(save=True); run_surgery_meds(save=True)
# merged_final = run_merge_surgery_meds(save=True)

---
# Table 12 — בדיקות מעבדה

| # | פעולה | סטטוס |
|---|--------|--------|
| 1 | CSV ללא header — `LAB_COLUMNS` | ✅ |
| 2 | Drop שורות ללא `מספר מקרה` | ✅ |
| 3 | לכל `(מספר מקרה, תאריך ניתוח)` — `תאריך בדיקה` הקרוב ביותר | ✅ |
| 4 | Pivot: 53 בדיקות נבחרות בלבד (`LAB_SELECTED_TESTS`) | ✅ |
| 5 | חסר → `0` | ✅ |

In [27]:
labs = run_labs(save=False)
print(f"unique tests: {labs[LAB_TEST_COL].nunique()}")
labs.head(3)

extract: (3715761, 6)
clean:   (3651411, 6)
unique tests: 53


,patient,מספר מקרה,קטגוריית בדיקה,שם בדיקה,ערך,תאריך בדיקה
0,100020,11529175,ביוכימיה - כימיה בדם,BUN-B,10.60,2022-10-02 03:19:00
1,100020,11529175,ביוכימיה - כימיה בדם,Creatinine-B,1.34,2022-10-02 03:19:00
2,100020,11529175,ביוכימיה - כימיה בדם,CRP,0.70,2022-10-02 03:19:00


In [28]:
merged_final = merge_labs(merged_with_smoking, labs)
merged_final = prepare_final_dataset(merged_final)
merged_final = finalize_english_ml_dataset(merged_final)
print(f"shape: {merged_final.shape}")
print(f"columns sample: {list(merged_final.columns[:12])} ... {list(merged_final.columns[-5:])}")
merged_final.head(3)

lab columns: 53
rows with lab data: 20,071 / 34,359
shape: (34359, 179)
columns sample: ['Patient', 'Medical_Record', 'Case_Number', 'Surgical_Department', 'Surgery_Date', 'Age', 'Male', 'Female', 'BMI', 'Normalized_BMI', 'Procedure_Code', 'Procedure_Side'] ... ['RBC_Z', 'RDW_Z', 'Sodium-B_Z', 'Uric acid-B_Z', 'WBC_Z']


,Patient,Medical_Record,Case_Number,Surgical_Department,Surgery_Date,Age,Male,Female,BMI,Normalized_BMI,Procedure_Code,Procedure_Side,Oxygen_Saturation,BP_Systolic_Before_Surgery,BP_Diastolic_Before_Surgery,Smoking,CT_Hospitalized,CT_Ambulatory,ST_Elective,ST_Emergency,ST_Non_Surgical_Operation,ST_Organ_Procurement,ST_Sesia,ST_Urgent,AT_General,AT_General_Local,AT_General_Regional,AT_Local,AT_Local_Regional,AT_Other,AT_Regional,AT_Sedation,AT_Spinal,AT_Spinal_Epidural,AT_Epidural,AT_Local_Sedation,AT_Regional_Sedation,Sunday,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Drug_Names,Allergy_Drug_Names,Disease_ICD9_Codes,Total_Surgery_Time,Estimated_Surgery_Time,Cleaned_Drug_Names,Cleaned_Allergy_Drug_Names,Num_Medications,Num_Allergy_Medications,Surgery_Time_Anomaly,Age_Group_Young,Age_Group_Middle-aged,Age_Group_Elderly,BMI_Category_Underweight,BMI_Category_Normal,BMI_Category_Overweight,BMI_Category_Obese,ICD9_272.4,ICD9_401.9,ICD9_401.1,ICD9_250,ICD9_414.9,ICD9_305.1,ICD9_278,ICD9_250.02,ICD9_427.31,ICD9_244.9,ICD9_250.9,ICD9_493.9,ICD9_278.01,ICD9_496,ICD9_285.9,ICD9_600,ICD9_V15.82,ICD9_733,ICD9_272.2,ICD9_585.9,ICD9_434.91,ICD9_571.8,ICD9_272,ICD9_V58.61,ICD9_V45.81,ICD9_443.9,ICD9_428,ICD9_V45.01,ICD9_585.3,ICD9_530.81,ICD9_345.9,ICD9_280.9,ICD9_V13.87,ICD9_V13.82,ICD9_574.2,ICD9_327.23,ICD9_424,ICD9_714,ICD9_V13.85,ICD9_562.1,ICD9_585.6,ICD9_174.9,ICD9_424.1,ICD9_311,ICD9_585.2,ICD9_428.9,ICD9_V45.11,ICD9_553.3,ICD9_274.9,ICD9_411.89,ICD9_414.8,ICD9_586,ICD9_600.9,ICD9_250.03,ICD9_V45.86,ICD9_780.57,ICD9_592,ICD9_V12.54,ICD9_733.09,ICD9_555.9,ICD9_V13.84,ICD9_428.1,ICD9_566,ICD9_153.9,ICD9_354,ALT(GPT)-B_Z,AST(GOT)-B_Z,Albumin-B_Z,Albumin/Globulin-B_Z,Alkaline Phosphatase-B_Z,Amylase-B_Z,BASO abs_Z,BASO%_Z,BUN-B_Z,"Bilirubin, total-B_Z",CK-B_Z,CRP_Z,Calcium-B_Z,Creatinine-B_Z,EOS abs_Z,EOS%_Z,GGT- Gamma Glutam.Trans.-B_Z,Globulin - blood_Z,Glucose-B_Z,HCT_Z,HGB_Z,Hemolysis Index -B_Z,Icteric Index-B_Z,LD Lactate dehydrogenase-B_Z,LYMPHO abs_Z,LYMPHO%_Z,Lipemic Index-B_Z,MCH_Z,MCHC_Z,MCV_Z,MONO abs_Z,MONO%_Z,MPV_Z,Magnesium-B_Z,NEUTRO abs_Z,NEUTRO%_Z,NRBC abs_Z,NRBC (Nucleated red blood cells) / leukocytes % - blood_Z,OSM-cal-B_Z,PCT_Z,PDW_Z,"PT,SEC_Z",PT-INR-B_Z,PTT-blood_Z,Phosphor -B_Z,"Platelet, automated count - blood_Z",Potassium-B_Z,"Protein, total-B_Z",RBC_Z,RDW_Z,Sodium-B_Z,Uric acid-B_Z,WBC_Z
0,443795,1338976,4093,כירורגיה,2022-03-27,55,1,0,29.320987,0.375555,"41.5,54.11,55.4",0.5,1.0,0.607743,-0.708896,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,"FRESH FROZEN PLASMA **BLOOD TRANSFUSION**,PROP...",NaN,250,81,NaN,"FRESH FROZEN PLASMA,PROPOFOL,MARCAINE,LACTATED...",<NA>,11,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-0.573009,0.058026,-5.626504,-3.903941,-1.242479,-0.144762,-0.420834,-0.523430,1.636631,-0.376355,0.019734,-0.495962,-12.390026,0.448865,-0.593120,-0.437143,-0.420749,-5.35525,2.123167,-0.104345,-0.069307,-9.380193,0.0,-2.074945,0.537308,0.417074,0.0,1.242048,0.308055,1.339517,0.138735,-0.115041,0.567944,-7.570052,-0.042179,-0.256079,-0.077103,-0.123492,1.455304,-0.386789,0.984882,-8.674117,-0.847874,-5.953031,-3.776937,-0.119793,-0.313213,-8.036723,-0.668425,-0.856302,-0.191916,-2.629809,0.126219
1,443795,1341489,4093,כירורגיה,2022-03-29,55,1,0,29.320987,0.375555,"54.11,54.61",0.0,1.0,0.607743,-0.708896,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,"FRESH FROZEN PLASMA **BLOOD TRANSFUSION**,PROP...",NaN,250,92,NaN,"FRESH FROZEN PLASMA,PROPOFOL,MARCAINE,LACTATED...",<NA>,11,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-0.573009,-0.649560,-5.626504,-3.903941,-1.242479,-1.129041,-1.603363,-1.408706,-1.505744,-0.850803,-0.215450,-0.562531,-12.390026,-1.149881,-1.104267,-0.965506,-0.420749,-5.35525,-2.424490,-1.732228,-0.873192,-9.380193,0.0,-2.074945,-1.045499,-1.414390,0.

In [29]:
# #כמה ערכים שאינם ריקים יש בכל מחלקה לעמודות בדיקות מעבדה

# LAB_COLUMNS1 = [
#     "ALT(GPT)-B", "AST(GOT)-B", "Albumin-B", "Albumin/Globulin-B",
#     "Alkaline Phosphatase-B", "Amylase-B", "BASO abs", "BASO%",
#     "BUN-B", "Bilirubin, total-B", "CK-B", "CRP", "Calcium-B",
#     "Creatinine-B", "EOS abs", "EOS%", "GGT- Gamma Glutam.Trans.-B",
#     "Globulin - blood", "Glucose-B", "HCT", "HGB", "Hemolysis Index -B",
#     "Icteric Index-B", "LD Lactate dehydrogenase-B", "LYMPHO abs",
#     "LYMPHO%", "Lipemic Index-B", "MCH", "MCHC", "MCV", "MONO abs",
#     "MONO%", "MPV", "Magnesium-B", "NEUTRO abs", "NEUTRO%",
#     "NRBC  abs",
#     "NRBC (Nucleated red blood cells) / leukocytes % - blood",
#     "OSM-cal-B", "PCT", "PDW", "PT,SEC", "PT-INR-B", "PTT-blood",
#     "Phosphor -B", "Platelet, automated count - blood", "Potassium-B",
#     "Protein, total-B", "RBC", "RDW", "Sodium-B", "Uric acid-B", "WBC"
# ]
# zero_counts_by_dept = (
#     merged_final
#     .groupby("מחלקה מנתחת")[LAB_COLUMNS1]
#     .apply(lambda df: (df != 0).sum())
# )

# zero_counts_by_dept

In [30]:
department_exports = export_ml_dataset_by_department(merged_final)
print(f"\n{SURGICAL_DEPARTMENT_COL} counts:")
print(merged_final[SURGICAL_DEPARTMENT_COL].value_counts())

אורטופדיה: 14,772 rows -> /home/gur/Documents/Ruppin/CSY3/Final Project/סמסטר ב/ml_data_pipeline/data/clean/export/surgeries_ml_ready_orthopedics.csv
כירורגיה: 14,258 rows -> /home/gur/Documents/Ruppin/CSY3/Final Project/סמסטר ב/ml_data_pipeline/data/clean/export/surgeries_ml_ready_general_surgery.csv
אף אוזן גרון: 5,329 rows -> /home/gur/Documents/Ruppin/CSY3/Final Project/סמסטר ב/ml_data_pipeline/data/clean/export/surgeries_ml_ready_ent.csv

Surgical_Department counts:
Surgical_Department
אורטופדיה       14772
כירורגיה        14258
אף אוזן גרון     5329
Name: count, dtype: int64


In [31]:
merged_final.head(20)

,Patient,Medical_Record,Case_Number,Surgical_Department,Surgery_Date,Age,Male,Female,BMI,Normalized_BMI,Procedure_Code,Procedure_Side,Oxygen_Saturation,BP_Systolic_Before_Surgery,BP_Diastolic_Before_Surgery,Smoking,CT_Hospitalized,CT_Ambulatory,ST_Elective,ST_Emergency,ST_Non_Surgical_Operation,ST_Organ_Procurement,ST_Sesia,ST_Urgent,AT_General,AT_General_Local,AT_General_Regional,AT_Local,AT_Local_Regional,AT_Other,AT_Regional,AT_Sedation,AT_Spinal,AT_Spinal_Epidural,AT_Epidural,AT_Local_Sedation,AT_Regional_Sedation,Sunday,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Drug_Names,Allergy_Drug_Names,Disease_ICD9_Codes,Total_Surgery_Time,Estimated_Surgery_Time,Cleaned_Drug_Names,Cleaned_Allergy_Drug_Names,Num_Medications,Num_Allergy_Medications,Surgery_Time_Anomaly,Age_Group_Young,Age_Group_Middle-aged,Age_Group_Elderly,BMI_Category_Underweight,BMI_Category_Normal,BMI_Category_Overweight,BMI_Category_Obese,ICD9_272.4,ICD9_401.9,ICD9_401.1,ICD9_250,ICD9_414.9,ICD9_305.1,ICD9_278,ICD9_250.02,ICD9_427.31,ICD9_244.9,ICD9_250.9,ICD9_493.9,ICD9_278.01,ICD9_496,ICD9_285.9,ICD9_600,ICD9_V15.82,ICD9_733,ICD9_272.2,ICD9_585.9,ICD9_434.91,ICD9_571.8,ICD9_272,ICD9_V58.61,ICD9_V45.81,ICD9_443.9,ICD9_428,ICD9_V45.01,ICD9_585.3,ICD9_530.81,ICD9_345.9,ICD9_280.9,ICD9_V13.87,ICD9_V13.82,ICD9_574.2,ICD9_327.23,ICD9_424,ICD9_714,ICD9_V13.85,ICD9_562.1,ICD9_585.6,ICD9_174.9,ICD9_424.1,ICD9_311,ICD9_585.2,ICD9_428.9,ICD9_V45.11,ICD9_553.3,ICD9_274.9,ICD9_411.89,ICD9_414.8,ICD9_586,ICD9_600.9,ICD9_250.03,ICD9_V45.86,ICD9_780.57,ICD9_592,ICD9_V12.54,ICD9_733.09,ICD9_555.9,ICD9_V13.84,ICD9_428.1,ICD9_566,ICD9_153.9,ICD9_354,ALT(GPT)-B_Z,AST(GOT)-B_Z,Albumin-B_Z,Albumin/Globulin-B_Z,Alkaline Phosphatase-B_Z,Amylase-B_Z,BASO abs_Z,BASO%_Z,BUN-B_Z,"Bilirubin, total-B_Z",CK-B_Z,CRP_Z,Calcium-B_Z,Creatinine-B_Z,EOS abs_Z,EOS%_Z,GGT- Gamma Glutam.Trans.-B_Z,Globulin - blood_Z,Glucose-B_Z,HCT_Z,HGB_Z,Hemolysis Index -B_Z,Icteric Index-B_Z,LD Lactate dehydrogenase-B_Z,LYMPHO abs_Z,LYMPHO%_Z,Lipemic Index-B_Z,MCH_Z,MCHC_Z,MCV_Z,MONO abs_Z,MONO%_Z,MPV_Z,Magnesium-B_Z,NEUTRO abs_Z,NEUTRO%_Z,NRBC abs_Z,NRBC (Nucleated red blood cells) / leukocytes % - blood_Z,OSM-cal-B_Z,PCT_Z,PDW_Z,"PT,SEC_Z",PT-INR-B_Z,PTT-blood_Z,Phosphor -B_Z,"Platelet, automated count - blood_Z",Potassium-B_Z,"Protein, total-B_Z",RBC_Z,RDW_Z,Sodium-B_Z,Uric acid-B_Z,WBC_Z
0,443795,1338976,4093,כירורגיה,2022-03-27,55,1,0,29.320987,0.375555,"41.5,54.11,55.4",0.5,1.00,0.607743,-0.708896,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,"FRESH FROZEN PLASMA **BLOOD TRANSFUSION**,PROP...",NaN,250,81,NaN,"FRESH FROZEN PLASMA,PROPOFOL,MARCAINE,LACTATED...",<NA>,11,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-0.573009,0.058026,-5.626504,-3.903941,-1.242479,-0.144762,-0.420834,-0.523430,1.636631,-0.376355,0.019734,-0.495962,-12.390026,0.448865,-0.593120,-0.437143,-0.420749,-5.355250,2.123167,-0.104345,-0.069307,-9.380193,0.0,-2.074945,0.537308,0.417074,0.0,1.242048,0.308055,1.339517,0.138735,-0.115041,0.567944,-7.570052,-0.042179,-0.256079,-0.077103,-0.123492,1.455304,-0.386789,0.984882,-8.674117,-0.847874,-5.953031,-3.776937,-0.119793,-0.313213,-8.036723,-0.668425,-0.856302,-0.191916,-2.629809,0.126219
1,443795,1341489,4093,כירורגיה,2022-03-29,55,1,0,29.320987,0.375555,"54.11,54.61",0.0,1.00,0.607743,-0.708896,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,"FRESH FROZEN PLASMA **BLOOD TRANSFUSION**,PROP...",NaN,250,92,NaN,"FRESH FROZEN PLASMA,PROPOFOL,MARCAINE,LACTATED...",<NA>,11,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-0.573009,-0.649560,-5.626504,-3.903941,-1.242479,-1.129041,-1.603363,-1.408706,-1.505744,-0.850803,-0.215450,-0.562531,-12.390026,-1.149881,-1.104267,-0.965506,-0.420749,-5.355250,-2.424490,-1.732228,-0.873192,-9.380193,0.0,-2.074945,-1.045499,-1.41439

### Final Dataset Preparation

| Step | Function | Output |
|------|----------|--------|
| Z-scores + OHE cleanup | `prepare_final_dataset` | `BMI_Z`, `{lab}_Z`, `systolic_bp_Z`, … |
| English + ICD9 OHE + reorder | `finalize_english_ml_dataset` | `surgeries_ml_ready` |

`finalize_english_ml_dataset`:
- Top-K ICD9 OHE from `מחלות_רקע` (`BG_TOP_K_ICD9 = 65`, configurable)
- Rename columns to English; drop raw BP/sat, procedure/diagnosis names, quality flags
- Placeholder `Estimated_Surgery_Time`; `Surgery_Time_Anomaly` from `duration_quality`
- Column order per ML schema (demographics → surgery → labs)
- Export by department → `data/clean/export/surgeries_ml_ready_{orthopedics|general_surgery|ent}.csv`

In [32]:
# labs = run_labs(save=True)
# merged_final = run_merge_labs(save=True)  # includes prepare_final_dataset